# Figure 1 | Vascular-centered digital NVU reconstruction

Publication-ready analysis notebook for the manuscript figure. Exploratory checks, scratch outputs, and analyses unrelated to the figure panels have been removed. Paths are repository-relative; set `NVU_PROJECT_ROOT` when running from another location.

Analysis scope:
- Vascular-field inference and representative spatial maps.
- Digital NVU reconstruction, communication-distance filtering, and vascular-cell density summaries.
- Endothelial and mural score summaries used for manuscript figure panels.

In [ ]:
# Repository-relative paths
PROJECT_ROOT <- Sys.getenv("NVU_PROJECT_ROOT", unset = normalizePath("..", mustWork = FALSE))
DATA_DIR <- file.path(PROJECT_ROOT, "data")
RESULTS_DIR <- file.path(PROJECT_ROOT, "results")
REFERENCE_DIR <- file.path(PROJECT_ROOT, "references")
FIGURE_DIR <- file.path(RESULTS_DIR, "figure1")
dir.create(FIGURE_DIR, recursive = TRUE, showWarnings = FALSE)

In [ ]:
library(Seurat)
library(ggplot2)
library (tidyverse)
library(RColorBrewer) 
library(cowplot)
#library(reshape)
library(deldir)

In [ ]:
# 增强版：添加更多信息和质控
library(Seurat)
library(dplyr)

samplelist <- c(
    "D01574B6", "D01574C4", "C01840B1", "C01834C3",
    "B03421D4", "B03421F4", "D03556C4", "D03556D4",
    "D04303A6", "D04303D1", "D03556E4", "D03556E6",
    "D04305A2", "D04305C6", "D01574A6", "D01574B1",
    "D03556F4", "D03556F6", "D01574C2", "D01574C6",
    "B03421A5", "B03421A6", "D03556D6", "D03556E2",
    "D04305A4", "D04305A6", "C04595E2", "C04595F1",
    "AD_1", "AD_2", "control_1", "control_2"
)

base_path <- '../results/05_nvu_analysis_results/Hip/200/'

# 读取分组信息
grouplist <- read.csv('../data/sample_group_mapping.csv')

seurat_list <- list()
failed_samples <- c()

for(sample in samplelist) {
    file_path <- paste0(base_path, sample, '/nvu_filtered.rds')
    
    if(file.exists(file_path)) {
        tryCatch({
            cat(paste("正在读取:", sample, "\n"))
            obj <- readRDS(file_path)
            
            # 添加sample_id
            obj$sample_id <- sample
            
            # 添加group信息
            group_info <- grouplist$group[grouplist$sample == sample]
            if(length(group_info) > 0) {
                obj$group <- group_info
            } else {
                obj$group <- "Unknown"
            }
            
            seurat_list[[sample]] <- obj
            cat(paste("  ✓ 成功 - 细胞数:", ncol(obj), "\n"))
            
        }, error = function(e) {
            cat(paste("  ✗ 失败:", e$message, "\n"))
            failed_samples <<- c(failed_samples, sample)
        })
    } else {
        cat(paste("  ✗ 文件不存在\n"))
        failed_samples <- c(failed_samples, sample)
    }
}

cat(paste("\n成功:", length(seurat_list), "个样本"))
cat(paste("\n失败:", length(failed_samples), "个样本"))
if(length(failed_samples) > 0) {
    cat("\n失败的样本:", paste(failed_samples, collapse = ", "), "\n")
}

# 合并
if(length(seurat_list) > 1) {
    cat("\n开始合并...\n")
    merged_data <- merge(
        x = seurat_list[[1]], 
        y = seurat_list[2:length(seurat_list)],
        add.cell.ids = names(seurat_list),
        project = "Hip_NVU_200"
    )
    
    cat(paste("✓ 合并完成！总细胞数:", ncol(merged_data), "\n"))
    
    # 保存
    saveRDS(merged_data, paste0(base_path, 'Hip_200_merged_all_samples.rds'))
    cat("✓ 已保存\n")
}

## Vascular-field spatial maps

In [ ]:
library(ggplot2)
library(sf)
library(RANN)  # Seurat依赖，应该已安装

sample <- 'D01574B6'# D01574B6  D01574B1


# ===== 读取数据 =====
data <- readRDS(paste0('../results/05_nvu_analysis_results/Hip/200/', sample, '/nvu_metadata.rds'))
line <- read.csv(paste0('../data/line/Hip/', sample, '_line_coordinates.csv'))
meta <- read.csv(paste0('../results/01_Ficture/Hip/', sample, '/hex_vascular_18um.filter.csv'))

color <- c('#F7FBFF', '#DEEBF7', '#C6DBEF', '#9ECAE1', '#6BAED6', '#4292C6', '#2171B5', '#08519C', '#08306B')

# ===== 1. 创建凸包边界 =====
data_sf <- st_as_sf(data, coords = c("x", "y"))
hull <- st_convex_hull(st_union(data_sf))

# ===== 2. 筛选凸包内的meta点 =====
meta_sf <- st_as_sf(meta, coords = c("x", "y"))
inside <- st_intersects(meta_sf, hull, sparse = FALSE)[, 1]
meta_in <- meta[inside, ]

cat("筛选后数据点数:", nrow(meta_in), "\n")

# ===== 3. 自动检测六边形网格参数 =====
set.seed(42)
n_sub <- min(1000, nrow(meta_in))
pts <- as.matrix(meta_in[sample(nrow(meta_in), n_sub), c("x", "y")])
dm <- as.matrix(dist(pts))
diag(dm) <- Inf
nn_dist <- median(apply(dm, 1, min))  # 最近邻中位距离
hex_r <- nn_dist / sqrt(3)            # 六边形外接圆半径

cat("最近邻距离:", nn_dist, " | 六边形半径:", hex_r, "\n")

# ===== 4. 生成完整六边形网格（用于填充空白区域）=====
bbox <- st_bbox(hull)
# Pointy-top 六边形参数
dx <- sqrt(3) * hex_r    # 列间距
dy <- 1.5 * hex_r        # 行间距

x_seq <- seq(bbox["xmin"] - dx * 2, bbox["xmax"] + dx * 2, by = dx)
y_seq <- seq(bbox["ymin"] - dy * 2, bbox["ymax"] + dy * 2, by = dy)

# 生成网格：偶数行在x方向偏移 dx/2
grid_all <- do.call(rbind, lapply(seq_along(y_seq), function(i) {
  offset <- if (i %% 2 == 0) dx / 2 else 0
  data.frame(x = x_seq + offset, y = y_seq[i])
}))

# 只保留凸包内的网格点
grid_sf <- st_as_sf(grid_all, coords = c("x", "y"))
grid_in <- grid_all[st_intersects(grid_sf, hull, sparse = FALSE)[, 1], ]

cat("网格总点数:", nrow(grid_in), "\n")

# nn_match <- nn2(
#   data  = as.matrix(meta_in[, c("x", "y")]),
#   query = as.matrix(grid_in[, c("x", "y")]),
#   k = 1
# )
# threshold <- nn_dist * 0.5
# is_gap <- nn_match$nn.dists[, 1] >= threshold

# gaps <- grid_in[is_gap, ]
# gaps$Vascular_score_norm <- 0

# cat("空白填充点数:", nrow(gaps), "\n")

# plot_data <- rbind(
#   meta_in[, c("x", "y", "Vascular_score_norm")],
#   gaps[, c("x", "y", "Vascular_score_norm")]
# )

# ===== 5. 统一使用规则网格（替换原来的合并逻辑）=====
nn_match <- nn2(
  data  = as.matrix(meta_in[, c("x", "y")]),
  query = as.matrix(grid_in[, c("x", "y")]),
  k = 1
)
threshold <- nn_dist * 0.5

# 所有点都在规则网格上，近的赋数据值，远的赋0
grid_in$Vascular_score_norm <- ifelse(
  nn_match$nn.dists[, 1] < threshold,
  meta_in$Vascular_score_norm[nn_match$nn.idx[, 1]],
  0
)

plot_data <- grid_in   # <<< 只用网格坐标，不再rbind原始数据

# ===== 6. 向量化生成六边形多边形 =====
# Pointy-top 六边形顶点角度: 30°, 90°, 150°, 210°, 270°, 330°
# 如果需要 Flat-top，改为: 0°, 60°, 120°, 180°, 240°, 300°
angles <- seq(30, 330, by = 60) * pi / 180   # <<< pointy-top
# angles <- seq(0, 300, by = 60) * pi / 180  # <<< flat-top (备选)

nv <- length(angles)
n  <- nrow(plot_data)

# 按score排序，低分先画（底层），高分后画（上层覆盖）
plot_data <- plot_data[order(plot_data$Vascular_score_norm), ]

hex_df <- data.frame(
  px    = rep(plot_data$x, each = nv) + hex_r * rep(cos(angles), n),
  py    = rep(plot_data$y, each = nv) + hex_r * rep(sin(angles), n),
  id    = rep(seq_len(n), each = nv),
  score = rep(plot_data$Vascular_score_norm, each = nv)
)

# ===== 7. 分层绘图：灰色背景 + 蓝色渐变 =====
hex_bg <- hex_df[hex_df$score == 0, ]   # 空白区域 → 灰色
hex_fg <- hex_df[hex_df$score > 0, ]    # 有数据区域 → 蓝色渐变

p <- ggplot() +
  # 底层：灰色六边形（空白区域）
  geom_polygon(
    data = hex_bg,
    aes(x = px, y = py, group = id),
    fill = "white", color = "white", linewidth = 0.05
  ) +
  # 中层：蓝色渐变六边形（有数据区域）
  geom_polygon(
    data = hex_fg,
    aes(x = px, y = py, group = id, fill = score),
    color = "white", linewidth = 0.05   # 白色细边框区分相邻六边形
  ) +
  # 顶层：解剖边界线
  geom_point(
    data = line,
    aes(x = coor_x, y = coor_y),
    color = "black", size = 0.001, stroke = 0.1
  ) +
  # 颜色标度
  scale_fill_gradientn(
    colors = color,
    limits = c(0, 1),
    breaks = seq(0, 1, 0.25),
    name = "Vascular\nScore"
  ) +
  coord_fixed() +
  theme_void() +
  theme(
    legend.position = "right",
    legend.title = element_text(size = 12),
    legend.text  = element_text(size = 10)
  )

p

ggsave(
  paste0('../results/Figure1/Hip_', sample, '-vasularscore_hex_空间分布.pdf'),
  plot = p, height = 10, width = 10
)

In [ ]:
library(ggplot2)
library(sf)
library(RANN)
library(cowplot)

# 路径配置
data_dir    <- "../results/05_nvu_analysis_results/Hip/200/"
line_dir    <- "../data/line/Hip/"
ficture_dir <- "../results/01_Ficture/Hip/"
output_dir  <- "../results/Figure1/"

if (!dir.exists(output_dir)) dir.create(output_dir, recursive = TRUE)

color <- c('#F7FBFF', '#DEEBF7', '#C6DBEF', '#9ECAE1', '#6BAED6', '#4292C6', '#2171B5', '#08519C', '#08306B')

# 样本列表（加上D01574B1和D01574B6）
ad_samples <- c(
  'AD_1', 'AD_2',
   'D01574B6',   # <<< 新增
  "D01574C4",
  "C01834C3", "C01840B1",
  "B03421F4", "B03421D4",
  "D03556C4", "D03556D4",
  "D04303A6", "D04303D1",
  "D03556E4", "D03556E6",
  "D04305A2", "D04305C6"
)

control_samples <- c(
  'control_1', 'control_2',
  "D01574A6",'D01574B1',
  "D01574C2", "D01574C6",
  "B03421A5", "B03421A6",
  "D03556D6", "D03556E2",
  "D04305A6", "D04305A4",
  "C04595E2", "C04595F1",
  "D03556F6", "D03556F4"
)

# 单样本 Hex Vascular Score 绘图函数
plot_hex_vascular <- function(sample_name) {
  
  # --- 读取数据 ---
  data_path <- paste0(data_dir, sample_name, "/nvu_metadata.rds")
  line_path <- paste0(line_dir, sample_name, "_line_coordinates.csv")
  meta_path <- paste0(ficture_dir, sample_name, "/hex_vascular_18um.filter.csv")
  
  if (!file.exists(data_path)) stop("nvu_metadata.rds not found")
  if (!file.exists(line_path)) stop("line_coordinates.csv not found")
  if (!file.exists(meta_path)) stop("hex_vascular_18um.filter.csv not found")
  
  data <- readRDS(data_path)
  line <- read.csv(line_path)
  meta <- read.csv(meta_path)
  
  # --- 1. 凸包 ---
  data_sf <- st_as_sf(data, coords = c("x", "y"))
  hull <- st_convex_hull(st_union(data_sf))
  
  # --- 2. 筛选凸包内meta ---
  meta_sf <- st_as_sf(meta, coords = c("x", "y"))
  inside <- st_intersects(meta_sf, hull, sparse = FALSE)[, 1]
  meta_in <- meta[inside, ]
  
  if (nrow(meta_in) < 10) stop("Too few points after filtering")
  
  # --- 3. 六边形参数 ---
  set.seed(42)
  n_sub <- min(1000, nrow(meta_in))
  pts <- as.matrix(meta_in[sample(nrow(meta_in), n_sub), c("x", "y")])
  dm <- as.matrix(dist(pts))
  diag(dm) <- Inf
  nn_dist <- median(apply(dm, 1, min))
  hex_r <- nn_dist / sqrt(3)
  
  # --- 4. 生成规则网格 ---
  bbox <- st_bbox(hull)
  dx <- sqrt(3) * hex_r
  dy <- 1.5 * hex_r
  
  x_seq <- seq(bbox["xmin"] - dx * 2, bbox["xmax"] + dx * 2, by = dx)
  y_seq <- seq(bbox["ymin"] - dy * 2, bbox["ymax"] + dy * 2, by = dy)
  
  grid_all <- do.call(rbind, lapply(seq_along(y_seq), function(i) {
    offset <- if (i %% 2 == 0) dx / 2 else 0
    data.frame(x = x_seq + offset, y = y_seq[i])
  }))
  
  grid_sf <- st_as_sf(grid_all, coords = c("x", "y"))
  grid_in <- grid_all[st_intersects(grid_sf, hull, sparse = FALSE)[, 1], ]
  
  # --- 5. 映射score到网格 ---
  nn_match <- nn2(
    data  = as.matrix(meta_in[, c("x", "y")]),
    query = as.matrix(grid_in[, c("x", "y")]),
    k = 1
  )
  threshold <- nn_dist * 0.5
  
  grid_in$Vascular_score_norm <- ifelse(
    nn_match$nn.dists[, 1] < threshold,
    meta_in$Vascular_score_norm[nn_match$nn.idx[, 1]],
    0
  )
  
  plot_data <- grid_in
  
  # --- 6. 生成六边形多边形 ---
  angles <- seq(30, 330, by = 60) * pi / 180
  nv <- 6
  n  <- nrow(plot_data)
  
  plot_data <- plot_data[order(plot_data$Vascular_score_norm), ]
  
  hex_df <- data.frame(
    px    = rep(plot_data$x, each = nv) + hex_r * rep(cos(angles), n),
    py    = rep(plot_data$y, each = nv) + hex_r * rep(sin(angles), n),
    id    = rep(seq_len(n), each = nv),
    score = rep(plot_data$Vascular_score_norm, each = nv)
  )
  
  hex_bg <- hex_df[hex_df$score == 0, ]
  hex_fg <- hex_df[hex_df$score > 0, ]
  
  # --- 7. 绘图 ---
  p <- ggplot() +
    geom_polygon(data = hex_bg, aes(x = px, y = py, group = id),
                 fill = "white", color = "white", linewidth = 0.05) +
    geom_polygon(data = hex_fg, aes(x = px, y = py, group = id, fill = score),
                 color = "white", linewidth = 0.05) +
    geom_point(data = line, aes(x = coor_x, y = coor_y),
               color = "black", size = 0.001, stroke = 0.1) +
    scale_fill_gradientn(
      colors = color, limits = c(0, 1),
      breaks = seq(0, 1, 0.25), name = "Vascular\nScore"
    ) +
    coord_fixed() +
    theme_void() +
    theme(legend.position = "none",
          plot.background = element_rect(fill = "white", color = NA),
          panel.background = element_rect(fill = "white", color = NA),
          plot.title = element_text(size = 8, face = "bold", hjust = 0.5,
                                    margin = margin(b = 2)),
          plot.margin = margin(2, 2, 2, 2))
  
  # 清理内存
  rm(data, line, meta, meta_in, grid_all, grid_in, hex_df, hex_bg, hex_fg, plot_data)
  gc()
  
  return(p)
}

# 批量生成
cat(rep("=", 70), "\n", sep = "")
cat("HIP HEX VASCULAR SCORE - ALL SAMPLES\n")
cat(rep("=", 70), "\n", sep = "")

generate_hex_plots <- function(sample_list, group_name) {
  cat("\nProcessing", group_name, "(", length(sample_list), "samples)...\n")
  plot_list <- list()
  
  for (sample in sample_list) {
    tryCatch({
      p <- plot_hex_vascular(sample)
      plot_list[[sample]] <- p + ggtitle(sample)
      cat("  ✓", sample, "\n")
    }, error = function(e) {
      cat("  ✗", sample, ":", e$message, "\n")
      plot_list[[sample]] <<- ggplot() +
        annotate("text", x = 0.5, y = 0.5, label = paste("Failed:", sample), size = 3) +
        theme_void() +
        theme(plot.background = element_rect(fill = "white", color = NA))
    })
  }
  return(plot_list)
}

ad_plots      <- generate_hex_plots(ad_samples, "AD")
control_plots <- generate_hex_plots(control_samples, "Control")

# 共享图例
legend_plot <- ggplot(data.frame(x = 1, y = 1, score = 0.5), aes(x, y, fill = score)) +
  geom_tile() +
  scale_fill_gradientn(
    colors = color, limits = c(0, 1),
    breaks = seq(0, 1, 0.25), name = "Vascular Score"
  ) +
  theme_void() +
  theme(legend.position = "right",
        legend.title = element_text(size = 10, face = "bold"),
        legend.text  = element_text(size = 9))

shared_legend <- get_legend(legend_plot)

# 组装 Grid（每行5个）
ad_rows      <- ceiling(length(ad_plots) / 5)
control_rows <- ceiling(length(control_plots) / 5)

ad_grid <- plot_grid(plotlist = ad_plots, ncol = 5, align = "hv", axis = "tblr")
control_grid <- plot_grid(plotlist = control_plots, ncol = 5, align = "hv", axis = "tblr")

ad_with_label <- plot_grid(
  ggdraw() + draw_label("AD", fontface = "bold", size = 14) +
    theme(plot.background = element_rect(fill = "white", color = NA)),
  ad_grid,
  ncol = 1, rel_heights = c(0.04, 1)
)

control_with_label <- plot_grid(
  ggdraw() + draw_label("Control", fontface = "bold", size = 14) +
    theme(plot.background = element_rect(fill = "white", color = NA)),
  control_grid,
  ncol = 1, rel_heights = c(0.04, 1)
)

# 主体 + 图例
main_body <- plot_grid(
  ad_with_label,
  control_with_label,
  ncol = 1,
  rel_heights = c(ad_rows, control_rows)
)

final <- plot_grid(
  main_body, shared_legend,
  ncol = 2, rel_widths = c(1, 0.08)
)

final <- ggdraw(final) +
  theme(plot.background = element_rect(fill = "white", color = NA))

# 保存
total_height <- (ad_rows + control_rows) * 8 + 2

ggsave(
  paste0(output_dir, "Hip_all_hex_vascular_by_group.pdf"),
  final, width = 40, height = total_height, bg = "white", limitsize = FALSE
)

ggsave(
  paste0(output_dir, "Hip_all_hex_vascular_by_group.png"),
  final, width = 40, height = total_height, dpi = 300, bg = "white", limitsize = FALSE
)

cat("\n✓ Done! Saved to", output_dir, "\n")

In [ ]:
library(ggplot2)
library(sf)
library(RANN)

sample <- 'GSM8330069_B01809A3'  # GSM8330063_A02092E1   GSM8330069_B01809A3
sample <- 'GSM8330067_B01809C2'  
# ===== 读取数据 =====
data <- readRDS(paste0('../results/05_nvu_analysis_results/Cortex/200/', sample, '/nvu_metadata.rds'))
line <- read.csv(paste0('../data/line/Cortex/', sample, '_line_coordinates.csv'))
meta <- read.csv(paste0('../results/01_Ficture/Cortex/', sample, '/hex_vascular_18um.filter.csv'))

color <- c('#F7FBFF', '#DEEBF7', '#C6DBEF', '#9ECAE1', '#6BAED6', '#4292C6', '#2171B5', '#08519C', '#08306B')

# ===== 1. 创建凸包边界 =====
data_sf <- st_as_sf(data, coords = c("x", "y"))
hull <- st_convex_hull(st_union(data_sf))

# ===== 2. 筛选凸包内的meta点 =====
meta_sf <- st_as_sf(meta, coords = c("x", "y"))
inside <- st_intersects(meta_sf, hull, sparse = FALSE)[, 1]
meta_in <- meta[inside, ]

cat("筛选后数据点数:", nrow(meta_in), "\n")

# ===== 3. 自动检测六边形网格参数 =====
set.seed(42)
n_sub <- min(1000, nrow(meta_in))
pts <- as.matrix(meta_in[sample(nrow(meta_in), n_sub), c("x", "y")])
dm <- as.matrix(dist(pts))
diag(dm) <- Inf
nn_dist <- median(apply(dm, 1, min))
hex_r <- nn_dist / sqrt(3)

cat("最近邻距离:", nn_dist, " | 六边形半径:", hex_r, "\n")

# ===== 4. 生成完整六边形网格 =====
bbox <- st_bbox(hull)
dx <- sqrt(3) * hex_r
dy <- 1.5 * hex_r

x_seq <- seq(bbox["xmin"] - dx * 2, bbox["xmax"] + dx * 2, by = dx)
y_seq <- seq(bbox["ymin"] - dy * 2, bbox["ymax"] + dy * 2, by = dy)

grid_all <- do.call(rbind, lapply(seq_along(y_seq), function(i) {
  offset <- if (i %% 2 == 0) dx / 2 else 0
  data.frame(x = x_seq + offset, y = y_seq[i])
}))

grid_sf <- st_as_sf(grid_all, coords = c("x", "y"))
grid_in <- grid_all[st_intersects(grid_sf, hull, sparse = FALSE)[, 1], ]

cat("网格总点数:", nrow(grid_in), "\n")

# ===== 5. 统一使用规则网格，映射score =====
nn_match <- nn2(
  data  = as.matrix(meta_in[, c("x", "y")]),
  query = as.matrix(grid_in[, c("x", "y")]),
  k = 1
)
threshold <- nn_dist * 0.5

grid_in$Vascular_score_norm <- ifelse(
  nn_match$nn.dists[, 1] < threshold,
  meta_in$Vascular_score_norm[nn_match$nn.idx[, 1]],
  0
)

plot_data <- grid_in

# ===== 6. 生成六边形多边形 =====
angles <- seq(30, 330, by = 60) * pi / 180
nv <- length(angles)
n  <- nrow(plot_data)

plot_data <- plot_data[order(plot_data$Vascular_score_norm), ]

hex_df <- data.frame(
  px    = rep(plot_data$x, each = nv) + hex_r * rep(cos(angles), n),
  py    = rep(plot_data$y, each = nv) + hex_r * rep(sin(angles), n),
  id    = rep(seq_len(n), each = nv),
  score = rep(plot_data$Vascular_score_norm, each = nv)
)

hex_bg <- hex_df[hex_df$score == 0, ]
hex_fg <- hex_df[hex_df$score > 0, ]

# ===== 7. 绘图 =====
p <- ggplot() +
  geom_polygon(
    data = hex_bg, aes(x = px, y = py, group = id),
    fill = "white", color = "white", linewidth = 0.05
  ) +
  geom_polygon(
    data = hex_fg, aes(x = px, y = py, group = id, fill = score),
    color = "white", linewidth = 0.05
  ) +
  geom_point(
    data = line, aes(x = x_new, y = y_new),
    color = "black", size = 0.001, stroke = 0.1
  ) +
  scale_fill_gradientn(
    colors = color, limits = c(0, 1),
    breaks = seq(0, 1, 0.25), name = "Vascular\nScore"
  ) +
  coord_fixed() +
  theme_void() +
  theme(
    legend.position = "right",
    legend.title = element_text(size = 12),
    legend.text  = element_text(size = 10)
  )

p

ggsave(
  paste0('../results/Figure1/Cortex_', sample, '-vasularscore_hex_空间分布.pdf'),
  plot = p, height = 10, width = 10
)

In [ ]:
library(ggplot2)
library(sf)
library(RANN)
library(cowplot)

# 路径配置
data_dir    <- "../results/05_nvu_analysis_results/Cortex/200/"
line_dir    <- "../data/line/Cortex/"
ficture_dir <- "../results/01_Ficture/Cortex/"
output_dir  <- "../results/Figure1/"

if (!dir.exists(output_dir)) dir.create(output_dir, recursive = TRUE)

color <- c('#F7FBFF', '#DEEBF7', '#C6DBEF', '#9ECAE1', '#6BAED6', '#4292C6', '#2171B5', '#08519C', '#08306B')

samples_list <- c(
  "GSM8330066_D02175A4",
  "GSM8330068_B01809A4",
  "GSM8330070_D02175A6",
  "GSM8330069_B01809A3",
  "GSM8330071_B01806B5",
  "GSM8330065_B01806B6",
  "GSM8330060_B02009F6",
  "GSM8330061_B02008D2",
  "GSM8330062_C02248B5",
  "GSM8330063_A02092E1",
  "GSM8330064_B02008C6"
)

# 单样本 Hex Vascular Score 绘图函数（Cortex版：line用x_new/y_new）
plot_hex_vascular_cortex <- function(sample_name) {
  
  data <- readRDS(paste0(data_dir, sample_name, "/nvu_metadata.rds"))
  line <- read.csv(paste0(line_dir, sample_name, "_line_coordinates.csv"))
  meta <- read.csv(paste0(ficture_dir, sample_name, "/hex_vascular_18um.filter.csv"))
  
  # 1. 凸包
  data_sf <- st_as_sf(data, coords = c("x", "y"))
  hull <- st_convex_hull(st_union(data_sf))
  
  # 2. 筛选
  meta_sf <- st_as_sf(meta, coords = c("x", "y"))
  inside <- st_intersects(meta_sf, hull, sparse = FALSE)[, 1]
  meta_in <- meta[inside, ]
  if (nrow(meta_in) < 10) stop("Too few points")
  
  # 3. 六边形参数
  set.seed(42)
  n_sub <- min(1000, nrow(meta_in))
  pts <- as.matrix(meta_in[sample(nrow(meta_in), n_sub), c("x", "y")])
  dm <- as.matrix(dist(pts))
  diag(dm) <- Inf
  nn_dist <- median(apply(dm, 1, min))
  hex_r <- nn_dist / sqrt(3)
  
  # 4. 规则网格
  bbox <- st_bbox(hull)
  dx <- sqrt(3) * hex_r
  dy <- 1.5 * hex_r
  
  x_seq <- seq(bbox["xmin"] - dx * 2, bbox["xmax"] + dx * 2, by = dx)
  y_seq <- seq(bbox["ymin"] - dy * 2, bbox["ymax"] + dy * 2, by = dy)
  
  grid_all <- do.call(rbind, lapply(seq_along(y_seq), function(i) {
    offset <- if (i %% 2 == 0) dx / 2 else 0
    data.frame(x = x_seq + offset, y = y_seq[i])
  }))
  
  grid_sf <- st_as_sf(grid_all, coords = c("x", "y"))
  grid_in <- grid_all[st_intersects(grid_sf, hull, sparse = FALSE)[, 1], ]
  
  # 5. 映射score
  nn_match <- nn2(
    data  = as.matrix(meta_in[, c("x", "y")]),
    query = as.matrix(grid_in[, c("x", "y")]),
    k = 1
  )
  threshold <- nn_dist * 0.5
  grid_in$Vascular_score_norm <- ifelse(
    nn_match$nn.dists[, 1] < threshold,
    meta_in$Vascular_score_norm[nn_match$nn.idx[, 1]],
    0
  )
  
  # 6. 六边形多边形
  angles <- seq(30, 330, by = 60) * pi / 180
  nv <- 6
  plot_data <- grid_in[order(grid_in$Vascular_score_norm), ]
  n <- nrow(plot_data)
  
  hex_df <- data.frame(
    px    = rep(plot_data$x, each = nv) + hex_r * rep(cos(angles), n),
    py    = rep(plot_data$y, each = nv) + hex_r * rep(sin(angles), n),
    id    = rep(seq_len(n), each = nv),
    score = rep(plot_data$Vascular_score_norm, each = nv)
  )
  
  hex_bg <- hex_df[hex_df$score == 0, ]
  hex_fg <- hex_df[hex_df$score > 0, ]
  
  # 7. 绘图（Cortex: x_new / y_new）
  p <- ggplot() +
    geom_polygon(data = hex_bg, aes(x = px, y = py, group = id),
                 fill = "white", color = "white", linewidth = 0.05) +
    geom_polygon(data = hex_fg, aes(x = px, y = py, group = id, fill = score),
                 color = "white", linewidth = 0.05) +
    geom_point(data = line, aes(x = x_new, y = y_new),
               color = "black", size = 0.001, stroke = 0.1) +
    scale_fill_gradientn(
      colors = color, limits = c(0, 1),
      breaks = seq(0, 1, 0.25), name = "Vascular\nScore"
    ) +
    coord_fixed() +
    theme_void() +
    theme(legend.position = "none",
          plot.background = element_rect(fill = "white", color = NA),
          panel.background = element_rect(fill = "white", color = NA),
          plot.title = element_text(size = 8, face = "bold", hjust = 0.5,
                                    margin = margin(b = 2)),
          plot.margin = margin(2, 2, 2, 2))
  
  rm(data, line, meta, meta_in, grid_all, grid_in, hex_df, hex_bg, hex_fg, plot_data)
  gc()
  
  return(p)
}

# 批量生成
cat(rep("=", 70), "\n", sep = "")
cat("CORTEX HEX VASCULAR SCORE - ALL SAMPLES\n")
cat(rep("=", 70), "\n", sep = "")

plot_list <- list()

for (sample in samples_list) {
  tryCatch({
    p <- plot_hex_vascular_cortex(sample)
    plot_list[[sample]] <- p + ggtitle(sample)
    cat("  ✓", sample, "\n")
  }, error = function(e) {
    cat("  ✗", sample, ":", e$message, "\n")
    plot_list[[sample]] <<- ggplot() +
      annotate("text", x = 0.5, y = 0.5, label = paste("Failed:", sample), size = 3) +
      theme_void() +
      theme(plot.background = element_rect(fill = "white", color = NA))
  })
}

# 共享图例
legend_plot <- ggplot(data.frame(x = 1, y = 1, score = 0.5), aes(x, y, fill = score)) +
  geom_tile() +
  scale_fill_gradientn(
    colors = color, limits = c(0, 1),
    breaks = seq(0, 1, 0.25), name = "Vascular Score"
  ) +
  theme_void() +
  theme(legend.position = "right",
        legend.title = element_text(size = 10, face = "bold"),
        legend.text  = element_text(size = 9))

shared_legend <- get_legend(legend_plot)

# 拼图（每行5个）
n_rows <- ceiling(length(plot_list) / 5)

combined <- plot_grid(
  plotlist = plot_list,
  ncol = 5,
  align = "hv",
  axis = "tblr"
)

final <- plot_grid(
  combined, shared_legend,
  ncol = 2, rel_widths = c(1, 0.08)
)

final <- ggdraw(final) +
  theme(plot.background = element_rect(fill = "white", color = NA))

# 保存
total_height <- n_rows * 8

ggsave(
  paste0(output_dir, "Cortex_all_hex_vascular.pdf"),
  final, width = 40, height = total_height, bg = "white", limitsize = FALSE
)

ggsave(
  paste0(output_dir, "Cortex_all_hex_vascular.png"),
  final, width = 40, height = total_height, dpi = 300, bg = "white", limitsize = FALSE
)

cat("\n✓ Done! Saved to", output_dir, "\n")

## Digital NVU spatial maps

## Vascular cells within vascular regions

In [ ]:
library(ggplot2)
library(sf)
library(RANN)

sample <- 'D01574B1'# D01574B6  D01574B1

# ===== 读取数据 =====
data <- readRDS(paste0('../results/05_nvu_analysis_results/Hip/200/', sample, '/nvu_metadata.rds'))
data <- subset(data, isTrue == 'Matched')
line <- read.csv(paste0('../data/line/Hip/', sample, '_line_coordinates.csv'))
meta <- read.csv(paste0('../results/01_Ficture/Hip/', sample, '/hex_vascular_18um.filter.csv'))

# 颜色设置
vascular_colors <- c('#F7FBFF', '#DEEBF7', '#C6DBEF', '#9ECAE1', '#6BAED6', '#4292C6', '#2171B5', '#08519C', '#08306B')
cell_colors <- c(
  "Endo"     = "#e72709",
  "Pericyte" = "#FF8C42",
  "VSMC"     = "#A8E6CF"
)
size_mapping <- c("Endo" = 1, "Pericyte" = 1, "VSMC" = 1)

# ===== 1. 创建凸包边界 =====
data_sf <- st_as_sf(data, coords = c("x", "y"))
hull <- st_convex_hull(st_union(data_sf))

# ===== 2. 筛选凸包内的meta点 =====
meta_sf <- st_as_sf(meta, coords = c("x", "y"))
inside <- st_intersects(meta_sf, hull, sparse = FALSE)[, 1]
meta_in <- meta[inside, ]
cat("原始点数:", nrow(meta), " | 筛选后:", nrow(meta_in), "\n")

# ===== 3. 检测六边形网格参数 =====
set.seed(42)
n_sub <- min(1000, nrow(meta_in))
pts <- as.matrix(meta_in[sample(nrow(meta_in), n_sub), c("x", "y")])
dm <- as.matrix(dist(pts))
diag(dm) <- Inf
nn_dist <- median(apply(dm, 1, min))
hex_r <- nn_dist / sqrt(3) * 1.02  # 微调避免缝隙

cat("最近邻距离:", nn_dist, " | 六边形半径:", hex_r, "\n")

# ===== 4. 生成完整规则六边形网格 =====
bbox <- st_bbox(hull)
dx <- sqrt(3) * hex_r
dy <- 1.5 * hex_r

x_seq <- seq(bbox["xmin"] - dx * 2, bbox["xmax"] + dx * 2, by = dx)
y_seq <- seq(bbox["ymin"] - dy * 2, bbox["ymax"] + dy * 2, by = dy)

grid_all <- do.call(rbind, lapply(seq_along(y_seq), function(i) {
  offset <- if (i %% 2 == 0) dx / 2 else 0
  data.frame(x = x_seq + offset, y = y_seq[i])
}))

grid_sf <- st_as_sf(grid_all, coords = c("x", "y"))
grid_in <- grid_all[st_intersects(grid_sf, hull, sparse = FALSE)[, 1], ]

# ===== 5. 统一用规则网格，映射score =====
nn_match <- nn2(
  data  = as.matrix(meta_in[, c("x", "y")]),
  query = as.matrix(grid_in[, c("x", "y")]),
  k = 1
)
threshold <- nn_dist * 0.5

grid_in$Vascular_score_norm <- ifelse(
  nn_match$nn.dists[, 1] < threshold,
  meta_in$Vascular_score_norm[nn_match$nn.idx[, 1]],
  0
)

# ===== 6. 生成六边形多边形 =====
angles <- seq(30, 330, by = 60) * pi / 180  # pointy-top
# angles <- seq(0, 300, by = 60) * pi / 180  # flat-top 备选

nv <- length(angles)
n  <- nrow(grid_in)

# 按score排序：低分先画，高分覆盖在上
grid_in <- grid_in[order(grid_in$Vascular_score_norm), ]

hex_df <- data.frame(
  px    = rep(grid_in$x, each = nv) + hex_r * rep(cos(angles), n),
  py    = rep(grid_in$y, each = nv) + hex_r * rep(sin(angles), n),
  id    = rep(seq_len(n), each = nv),
  score = rep(grid_in$Vascular_score_norm, each = nv)
)

# 分离灰色背景和有值六边形
hex_bg <- hex_df[hex_df$score == 0, ]
hex_fg <- hex_df[hex_df$score > 0, ]

# ===== 7. 绘制整合图 =====
p <- ggplot() +
  # 第1层：灰色六边形（空白区域背景）
  geom_polygon(
    data = hex_bg,
    aes(x = px, y = py, group = id),
    fill = "white", color = "white", linewidth = 0.05
  ) +
  # 第2层：蓝色渐变六边形（Vascular score）
  geom_polygon(
    data = hex_fg,
    aes(x = px, y = py, group = id, fill = score),
    color = "white", linewidth = 0.05
  ) +
  scale_fill_gradientn(
    colors = vascular_colors,
    limits = c(0, 1),
    breaks = seq(0, 1, 0.25),
    name = "Vascular\nScore"
  ) +
  # 第3层：解剖边界线
  geom_point(
    data = line,
    aes(x = coor_x, y = coor_y),
    color = "black",
    size = 0.000000000001, stroke = 0.1
  ) +
  # 第4层：细胞点（最上层，用color而非fill，和hex的fill不冲突）
  geom_point(
    data = data,
    aes(x = x, y = y, color = celltype_unit, size = celltype_unit),
    alpha = 0.6
  ) +
  scale_color_manual(
    values = cell_colors,
    name = "Cell Type"
  ) +
  scale_size_manual(values = size_mapping, guide = "none") +
  # 主题
  coord_fixed() +
  theme_void() +
  theme(
    panel.background  = element_rect(fill = 'white'),
    plot.background   = element_rect(fill = 'white'),
    legend.background = element_rect(fill = 'white'),
    legend.key        = element_rect(fill = 'white'),
    legend.text       = element_text(color = 'black', size = 10),
    legend.title      = element_text(color = 'black', size = 12)
  ) +
  guides(
    fill  = guide_colorbar(order = 1),                          # Vascular图例在前
    color = guide_legend(override.aes = list(size = 3), order = 2)  # 细胞图例在后
  )

print(p)

ggsave(
  paste0('../results/Figure1/Hip_', sample, '-vascular_combined_hex.pdf'),
  plot = p, height = 8, width = 8
)

In [ ]:
library(ggplot2)
library(sf)
library(RANN)

sample <- 'GSM8330061_B02008D2'  # GSM8330063_A02092E1   GSM8330069_B01809A3
# sample <- 'GSM8330069_B01809A3'  # GSM8330063_A02092E1   GSM8330069_B01809A3
# sample <- 'GSM8330063_A02092E1'  # GSM8330063_A02092E1   GSM8330069_B01809A3
# ===== 读取数据 =====
data <- readRDS(paste0('../results/05_nvu_analysis_results/Cortex/200/', sample, '/nvu_metadata.rds'))
data <- subset(data, isTrue == 'Matched')
line <- read.csv(paste0('../data/line/Cortex/', sample, '_line_coordinates.csv'))
meta <- read.csv(paste0('../results/01_Ficture/Cortex/', sample, '/hex_vascular_18um.filter.csv'))

# 颜色设置
vascular_colors <- c('#F7FBFF', '#DEEBF7', '#C6DBEF', '#9ECAE1', '#6BAED6', '#4292C6', '#2171B5', '#08519C', '#08306B')
cell_colors <- c(
  "Endo"     = "#e72709",
  "Pericyte" = "#FF8C42",
  "VSMC"     = "#FF8C42"
)
size_mapping <- c("Endo" = 1, "Pericyte" = 1, "VSMC" = 1)

# ===== 1. 创建凸包边界 =====
data_sf <- st_as_sf(data, coords = c("x", "y"))
hull <- st_convex_hull(st_union(data_sf))

# ===== 2. 筛选凸包内的meta点 =====
meta_sf <- st_as_sf(meta, coords = c("x", "y"))
inside <- st_intersects(meta_sf, hull, sparse = FALSE)[, 1]
meta_in <- meta[inside, ]
cat("原始点数:", nrow(meta), " | 筛选后:", nrow(meta_in), "\n")

# ===== 3. 检测六边形网格参数 =====
set.seed(42)
n_sub <- min(1000, nrow(meta_in))
pts <- as.matrix(meta_in[sample(nrow(meta_in), n_sub), c("x", "y")])
dm <- as.matrix(dist(pts))
diag(dm) <- Inf
nn_dist <- median(apply(dm, 1, min))
hex_r <- nn_dist / sqrt(3) * 1.02

cat("最近邻距离:", nn_dist, " | 六边形半径:", hex_r, "\n")

# ===== 4. 生成完整规则六边形网格 =====
bbox <- st_bbox(hull)
dx <- sqrt(3) * hex_r
dy <- 1.5 * hex_r

x_seq <- seq(bbox["xmin"] - dx * 2, bbox["xmax"] + dx * 2, by = dx)
y_seq <- seq(bbox["ymin"] - dy * 2, bbox["ymax"] + dy * 2, by = dy)

grid_all <- do.call(rbind, lapply(seq_along(y_seq), function(i) {
  offset <- if (i %% 2 == 0) dx / 2 else 0
  data.frame(x = x_seq + offset, y = y_seq[i])
}))

grid_sf <- st_as_sf(grid_all, coords = c("x", "y"))
grid_in <- grid_all[st_intersects(grid_sf, hull, sparse = FALSE)[, 1], ]

# ===== 5. 统一用规则网格，映射score =====
nn_match <- nn2(
  data  = as.matrix(meta_in[, c("x", "y")]),
  query = as.matrix(grid_in[, c("x", "y")]),
  k = 1
)
threshold <- nn_dist * 0.5

grid_in$Vascular_score_norm <- ifelse(
  nn_match$nn.dists[, 1] < threshold,
  meta_in$Vascular_score_norm[nn_match$nn.idx[, 1]],
  0
)

# ===== 6. 生成六边形多边形 =====
angles <- seq(30, 330, by = 60) * pi / 180
nv <- length(angles)
n  <- nrow(grid_in)

grid_in <- grid_in[order(grid_in$Vascular_score_norm), ]

hex_df <- data.frame(
  px    = rep(grid_in$x, each = nv) + hex_r * rep(cos(angles), n),
  py    = rep(grid_in$y, each = nv) + hex_r * rep(sin(angles), n),
  id    = rep(seq_len(n), each = nv),
  score = rep(grid_in$Vascular_score_norm, each = nv)
)

hex_bg <- hex_df[hex_df$score == 0, ]
hex_fg <- hex_df[hex_df$score > 0, ]

# ===== 7. 绘制整合图 =====
p <- ggplot() +
  geom_polygon(
    data = hex_bg, aes(x = px, y = py, group = id),
    fill = "white", color = "white", linewidth = 0.05
  ) +
  geom_polygon(
    data = hex_fg, aes(x = px, y = py, group = id, fill = score),
    color = "white", linewidth = 0.05
  ) +
  scale_fill_gradientn(
    colors = vascular_colors, limits = c(0, 1),
    breaks = seq(0, 1, 0.25), name = "Vascular\nScore"
  ) +
  geom_point(
    data = line, aes(x = x_new, y = y_new),
    color = "black", size = 0.000000000001, stroke = 0.1
  ) +
  geom_point(
    data = data,
    aes(x = x, y = y, color = celltype_unit, size = celltype_unit),
    alpha = 0.6
  ) +
  scale_color_manual(values = cell_colors, name = "Cell Type") +
  scale_size_manual(values = size_mapping, guide = "none") +
  coord_fixed() +
  theme_void() +
  theme(
    panel.background  = element_rect(fill = 'white'),
    plot.background   = element_rect(fill = 'white'),
    legend.background = element_rect(fill = 'white'),
    legend.key        = element_rect(fill = 'white'),
    legend.text       = element_text(color = 'black', size = 10),
    legend.title      = element_text(color = 'black', size = 12)
  ) +
  guides(
    fill  = guide_colorbar(order = 1),
    color = guide_legend(override.aes = list(size = 3), order = 2)
  )

print(p)

ggsave(
  paste0('../results/Figure1/Cortex_', sample, '-vascular_combined_hex.pdf'),
  plot = p, height = 8, width = 8
)

## Communication-distance filtering

In [ ]:
library(ggplot2)
library(dplyr)

# ===== 1. 遍历所有样本，计算过滤比例 =====
base_dirs <- c(
  "Hip"    = "../results/05_nvu_analysis_results/Hip/200",
  "Cortex" = "../results/05_nvu_analysis_results/Cortex/200"
)

results <- list()

for (region in names(base_dirs)) {
  sample_dirs <- list.dirs(base_dirs[region], recursive = FALSE, full.names = TRUE)
  
  for (d in sample_dirs) {
    rds_file <- file.path(d, "nvu_marked.rds")
    if (!file.exists(rds_file)) next
    
    sample_name <- basename(d)
    obj <- readRDS(rds_file)
    
    # ===== 关键修改：从Seurat对象提取metadata =====
    meta <- obj@meta.data
    
    # dist == TRUE 的细胞
    dist_true <- meta[meta$dist == TRUE, ]
    n_dist_true <- nrow(dist_true)
    
    if (n_dist_true == 0) next
    
    n_filtered <- sum(dist_true$passed_communication_filter == FALSE)
    n_passed   <- sum(dist_true$passed_communication_filter == TRUE)
    
    results[[length(results) + 1]] <- data.frame(
      region      = region,
      sample      = sample_name,
      dist_true   = n_dist_true,
      filtered    = n_filtered,
      passed      = n_passed,
      filter_rate = n_filtered / n_dist_true * 100
    )
    
    rm(obj); gc()  # 释放内存
  }
}

df <- do.call(rbind, results)

print(df[, c("region", "sample", "dist_true", "filtered", "passed", "filter_rate")])

# ===== 2. 按区域汇总 =====
df_summary <- df %>%
  group_by(region) %>%
  summarise(
    n_samples   = n(),
    mean_rate   = mean(filter_rate),
    sd_rate     = sd(filter_rate),
    se_rate     = sd(filter_rate) / sqrt(n()),
    total_dist  = sum(dist_true),
    total_filt  = sum(filtered),
    overall_rate = total_filt / total_dist * 100,
    .groups = "drop"
  )

cat("\n===== 区域汇总 =====\n")
print(df_summary)

# ===== 3. 绘图：柱状图 + 散点 =====
# 设置区域顺序
df$region <- factor(df$region, levels = c("Hip", "Cortex"))
df_summary$region <- factor(df_summary$region, levels = c("Hip", "Cortex"))

p <- ggplot() +
  # 柱状图：均值
  geom_bar(
    data = df_summary,
    aes(x = region, y = mean_rate, fill = region),
    stat = "identity", width = 0.6, alpha = 0.7
  ) +
  # 误差棒：均值 ± SE
  geom_errorbar(
    data = df_summary,
    aes(x = region, ymin = mean_rate - se_rate, ymax = mean_rate + se_rate),
    width = 0.2, linewidth = 0.8
  ) +
  # 散点：每个样本
  geom_jitter(
    data = df,
    aes(x = region, y = filter_rate),
    width = 0.15, size = 2, alpha = 0.8, color = "black"
  ) +
  # 标注均值
  geom_text(
    data = df_summary,
    aes(x = region, y = mean_rate + se_rate + 2,
        label = paste0(round(mean_rate, 1), "%")),
    size = 4.5, fontface = "bold"
  ) +
  scale_fill_manual(values = c("Hip" = "#4292C6", "Cortex" = "#EF8A62")) +
  labs(
    x = NULL,
    y = "Filtered by Communication\nStrength (%)",
    title = "Proportion of dist=TRUE cells filtered\nby communication strength"
  ) +
  theme_classic(base_size = 14) +
  theme(
    legend.position = "none",
    plot.title = element_text(hjust = 0.5, size = 13),
    axis.text.x = element_text(size = 14, face = "bold")
  ) +
  scale_y_continuous(expand = expansion(mult = c(0, 0.1)))

print(p)

ggsave(
  '../results/Figure1/dist_communication_filter_rate.pdf',
  plot = p, height = 6, width = 4
)
# ===== 4. 保存总meta =====
write.csv(
  df,
  '../results/Figure1/dist_communication_filter_meta.csv',
  row.names = FALSE
)

cat("\n已保存，共", nrow(df), "个样本\n")

In [ ]:
# ===== 3. 绘图：柱状图 + 散点 =====
# 设置区域顺序
df$region <- factor(df$region, levels = c("Hip", "Cortex"))
df_summary$region <- factor(df_summary$region, levels = c("Hip", "Cortex"))

p <- ggplot() +
  # 柱状图：均值
  geom_bar(
    data = df_summary,
    aes(x = region, y = mean_rate, fill = region),
    stat = "identity", width = 0.6, alpha = 0.7
  ) +
  # 误差棒：均值 ± SE
  geom_errorbar(
    data = df_summary,
    aes(x = region, ymin = mean_rate - se_rate, ymax = mean_rate + se_rate),
    width = 0.2, linewidth = 0.8
  ) +
  # 散点：每个样本
  # geom_jitter(
  #   data = df,
  #   aes(x = region, y = filter_rate),
  #   width = 0.15, size = 2, alpha = 0, color = "black"
  # ) +
  # 标注均值
  geom_text(
    data = df_summary,
    aes(x = region, y = mean_rate + se_rate + 2,
        label = paste0(round(mean_rate, 1), "%")),
    size = 4.5, fontface = "bold"
  ) +
  scale_fill_manual(values = c("Hip" = "#4292C6", "Cortex" = "#EF8A62")) +
  labs(
    x = NULL,
    y = "Filtered by Communication\nStrength (%)",
    title = "Proportion of dist=TRUE cells filtered\nby communication strength"
  ) +
  theme_classic(base_size = 14) +
  theme(
    legend.position = "none",
    plot.title = element_text(hjust = 0.5, size = 13),
    axis.text.x = element_text(size = 14, face = "bold")
  ) +
  scale_y_continuous(expand = expansion(mult = c(0, 0.1)))

print(p)
ggsave(
  '../results/Figure1/dist_communication_filter_rate.pdf',
  plot = p, height = 6, width = 4
)

## Vascular density and endothelial/mural score summaries

In [ ]:
samplelist = c(
    "D01574B6", "D01574C4", "C01840B1", "C01834C3",
    "B03421D4", "B03421F4", "D03556C4", "D03556D4",
    "D04303A6", "D04303D1", "D03556E4", "D03556E6",
    "D04305A2", "D04305C6", "D01574A6", "D01574B1",
    "D03556F4", "D03556F6", "D01574C2", "D01574C6",
    "B03421A5", "B03421A6", "D03556D6", "D03556E2",
    "D04305A4", "D04305A6", "C04595E2", "C04595F1",
    "AD_1", "AD_2", "control_1", "control_2"
  )
data = readRDS('../results/05_nvu_analysis_results/Hip/200/D01574B6/nvu_metadata.rds')

data = data[,c('area','area_m','isTrue','celltype','percent.mito','celltype_unit','celltype','unit_id','dist')]

In [ ]:
# 1. 定义样本列表
samplelist = c(
  "D01574B6", "D01574C4", "C01840B1", "C01834C3",
  "B03421D4", "B03421F4", "D03556C4", "D03556D4",
  "D04303A6", "D04303D1", "D03556E4", "D03556E6",
  "D04305A2", "D04305C6", "D01574A6", "D01574B1",
  "D03556F4", "D03556F6", "D01574C2", "D01574C6",
  "B03421A5", "B03421A6", "D03556D6", "D03556E2",
  "D04305A4", "D04305A6", "C04595E2", "C04595F1",
  "AD_1", "AD_2", "control_1", "control_2"
)

samplelist = c("GSM8330060_B02009F6", "GSM8330061_B02008D2", "GSM8330062_C02248B5",
              "GSM8330063_A02092E1", "GSM8330064_B02008C6", "GSM8330067_B01809C2",
               "GSM8330065_B01806B6", "GSM8330066_D02175A4", 
                   "GSM8330068_B01809A4", "GSM8330069_B01809A3",
                   "GSM8330070_D02175A6", "GSM8330071_B01806B5")

# 2. 定义基础路径和需要提取的列（注意：去重celltype列，避免重复选择）
base_path <- '../results/05_nvu_analysis_results/Cortex/200/'
# base_path <- '../results/05_nvu_analysis_results/Cortex/300/'

# base_path <- '../results/05_nvu_analysis_results/Hip/200/'
# base_path <- '../results/05_nvu_analysis_results/Hip/300/'

target_columns <- c('area','area_m','isTrue','celltype','percent.mito',
                    'celltype_unit','unit_id','dist')  # 去重重复的celltype列，保留1个即可

# 3. 初始化空列表，用于存储每个样本处理后的数据集
metadata_list <- list()

# 4. 批量遍历所有样本，读取并处理数据
for (sample_id in samplelist) {
  # 构建当前样本的rds文件完整路径
  rds_file_path <- paste0(base_path, sample_id, '/nvu_metadata.rds')
  
  tryCatch({
    # 步骤1：读取rds文件
    sample_data <- readRDS(rds_file_path)
    
    # 步骤2：提取指定列（确保列存在，避免报错）
    # 先筛选出样本数据中实际存在的目标列
    existing_columns <- intersect(target_columns, colnames(sample_data))
    missing_columns <- setdiff(target_columns, colnames(sample_data))
    
    # 若有缺失列，给出警告提示
    if (length(missing_columns) > 0) {
      warning(paste0("样本 ", sample_id, " 缺失以下列：", paste(missing_columns, collapse = ", ")))
    }
    
    # 提取有效列
    sample_data_subset <- sample_data[, existing_columns, drop = FALSE]
    
    # 步骤3：添加样本ID列（关键，用于区分不同样本的数据）
    sample_data_subset$sample_id <- sample_id
    
    # 步骤4：将处理后的样本数据存入列表
    metadata_list[[sample_id]] <- sample_data_subset
    
    # 打印进度提示
    message(paste0("成功处理样本：", sample_id))
    
  }, error = function(e) {
    # 捕获错误并提示，避免循环中断
    warning(paste0("处理样本 ", sample_id, " 失败：", e$message))
  })
}

# 5. 整合所有样本数据为一个大的数据框（核心：合并列表中的所有数据集）
# 方法1：使用do.call(rbind, ...)（base R，无需额外安装包）
combined_metadata <- do.call(rbind, metadata_list)

# 方法2：使用dplyr::bind_rows（tidyverse风格，更高效，支持不规则数据框）
# 若未安装dplyr，先执行：install.packages("dplyr")
# library(dplyr)
# combined_metadata <- bind_rows(metadata_list, .id = "sample_id_from_list")

# 6. 重置行名（可选，使行名连续有序）
rownames(combined_metadata) <- NULL

# 7. 查看整合结果
message(paste0("\n数据整合完成！总数据量：", nrow(combined_metadata), " 行，", ncol(combined_metadata), " 列"))
saveRDS(combined_metadata,'../results/05_nvu_analysis_results/Cortex/200/Cortex_combined_metadata.rds')
# saveRDS(combined_metadata,'../results/05_nvu_analysis_results/Cortex/300/Cortex_combined_metadata.rds')

In [ ]:
# 加载所需包
library(dplyr)
library(ggplot2)
library(tidyr)
library(ggpubr)

# 1. 读取海马体数据
celldata = readRDS('../results/05_nvu_analysis_results/Hip/200/Hip_combined_metadata.rds')
areadata = read.csv('../data/Hip_region_bin50_summary.csv')
areadata$area_m = areadata$area
areadata = areadata[areadata$area_m%in%c('CA1','CA2','CA3','CA4','FAS','SLRM','DG'),]
# 2. 数据预处理（核心：基于chip建立对应关系）
# 2.1 定义血管细胞类型
vascular_cell_types <- c("Endo", "Pericyte")

# 2.2 筛选血管细胞数据 + 关键：为celldata补充chip映射（核心修改1）
# 步骤1：先从areadata中提取chip与sample的对应关系（chip→sample）
chip_sample_map <- areadata %>%
  distinct(chip, sample) %>%
  filter(!is.na(chip) & !is.na(sample))

# 步骤2：筛选血管细胞数据，并关联chip信息
celldata_vascular <- celldata %>%
  filter(
    celltype_unit %in% vascular_cell_types  # 血管细胞仅对应isTrue=Matched，无需额外过滤
  ) %>%
  rename(
    celldata_sample = sample_id  # 重命名避免冲突，明确是celldata中的样本ID
  ) %>%
  filter(!is.na(area_m)) %>%
  left_join(chip_sample_map, by = c("celldata_sample" = "sample")) %>%  # 关联chip
  filter(!is.na(chip))  %>%
  # 将 Pericyte 和 VSMC 合并为 Mural（壁细胞）
  mutate(
    celltype_unit = case_when(
      celltype_unit %in% c("Pericyte", "VSMC") ~ "Mural",
      TRUE ~ celltype_unit
    )
  )

# 2.3 处理areadata：① 基于chip提取分组 ② 按「chip+area_m」汇总面积（核心修改2）
# 步骤1：构建chip-group映射（从chip列提取AD/Control分组，基于chip对应）
chip_group_info <- areadata %>%
  distinct(chip) %>%
  mutate(
    group = case_when(
      grepl("AD", chip) ~ "AD",  # 匹配AD芯片（如AD1.1）
      grepl("control|Con", chip, ignore.case = TRUE) ~ "Control",  # 匹配Control芯片
      TRUE ~ "Unknown"  # 未知分组（备用）
    )
  ) %>%
  filter(group != "Unknown")

# 步骤2：处理areadata，按「chip+area_m」分组汇总面积（确保唯一，基于chip对应）
chip_area_map <- areadata %>%
  filter(!is.na(area_m) & area_mm2 > 0) %>%  # 过滤无效脑区和面积为0的记录
  # 核心：按「chip+area_m」分组（替代原sample+area_m），确保基于chip对应
  group_by(chip, area_m) %>%
  summarise(
    sample_area_mm2 = sum(area_mm2),  # 单个chip-单个脑区（area_m）的总面积
    .groups = "drop"
  ) %>%
  # 合并分组信息（基于chip匹配）
  left_join(chip_group_info, by = "chip") %>%
  filter(!is.na(group)) %>%
  # 验证唯一性（可选，调试用）
  group_by(chip, area_m) %>%
  mutate(duplicate_flag = n()) %>%
  ungroup() %>%
  filter(duplicate_flag == 1) %>%
  select(-duplicate_flag)

chip_area_map <- chip_area_map %>%
  left_join(
    chip_sample_map %>% rename(celldata_sample = sample),
    by = "chip"
  )

# 2.4 合并celldata和面积/分组信息（核心：基于「chip+area_m」匹配，修改3）
celldata_with_group <- celldata_vascular %>%
  left_join(
    chip_area_map,
    by = c("celldata_sample",'chip', "area_m")  # 统一用chip+area_m匹配，确保对应关系准确
  ) %>%
  filter(!is.na(group))  # 过滤无法匹配分组的记录

# 3. 单个样本（chip层面）密度统计（核心：保留chip维度，即样本维度）
sample_area_cell_stats <- celldata_with_group %>%
  group_by(chip, group, area_m, celltype_unit) %>%
  summarise(
    cell_number = n(),  # 单个chip-脑区-细胞类型的细胞数
    sample_brain_area = unique(sample_area_mm2),  # 单个chip-脑区的面积（唯一值）
    .groups = "drop"
  ) %>%
  # 计算单个chip（样本）的血管细胞密度
  mutate(
    density_per_mm2 = cell_number / sample_brain_area
  )
write.csv(sample_area_cell_stats,'../results/Figure1/Hip_density.csv')
# 4. 计算分组统计量（均值、标准差、标准误，用于误差棒）
group_area_stats <- sample_area_cell_stats %>%
  group_by(group, area_m, celltype_unit) %>%
  summarise(
    mean_density = mean(density_per_mm2),
    sd_density = sd(density_per_mm2),
    se_density = sd(density_per_mm2) / sqrt(n()),
    n_samples = n(),  # 此处n_samples为每组每个脑区的chip数量（即有效样本数）
    .groups = "drop"
  )

# 5. 可视化：带误差棒的AD vs Con分组对比图（横坐标=area_m）
# 选择误差棒类型：sd_density（标准差）或 se_density（标准误）
error_type <- "se_density"

p_hip_error <- ggplot() +
  # 绘制单个chip（样本）数据点（展示个体差异）
  geom_jitter(
    data = sample_area_cell_stats,
    aes(x = area_m, y = density_per_mm2, color = group),
    position = position_jitterdodge(dodge.width = 0.9, jitter.width = 0.2),
    alpha = 0.6, size = 2
  ) +
  # 绘制分组均值柱状图
  geom_col(
    data = group_area_stats,
    aes(x = area_m, y = mean_density, fill = group),
    position = "dodge", alpha = 0.8, width = 0.9
  ) +
  # 绘制误差棒
  geom_errorbar(
    data = group_area_stats,
    aes(
      x = area_m,
      ymin = mean_density - .data[[error_type]],
      ymax = mean_density + .data[[error_type]],
      group = group
    ),
    position = position_dodge(width = 0.9),
    width = 0.2, color = "black", linewidth = 0.5
  ) +
  # 按细胞类型分面
  facet_wrap(~celltype_unit, scales = "free_y") +
  # 标签与主题设置
  labs(
    title = "Hip 脑区 AD vs Con 血管细胞密度对比（基于chip匹配，带误差棒）",
    x = "脑区（area_m）",
    y = paste0("细胞密度（个/平方毫米）\n（误差棒：", ifelse(error_type=="sd_density", "标准差", "标准误"), "）"),
    fill = "分组",
    color = "分组"
  ) +
  theme_bw() +
  theme(
    plot.title = element_text(hjust = 0.5, size = 14, face = "bold"),
    axis.text.x = element_text(angle = 45, hjust = 1, size = 10),
    axis.title = element_text(size = 11),
    legend.position = "top",
    strip.text = element_text(size = 10, face = "bold")
  )

# 6. 展示图形并保存
print(p_hip_error)

ggsave(
  "../results/figure1/Hip_AD_Con_density_comparison_with_error_bars_chip_matched.png",
  p_hip_error,
  width = 14, height = 8, dpi = 300
)

# 7. 验证结果（chip层面、分组样本数、脑区分布）
valid_chips <- sample_area_cell_stats %>% distinct(chip, group)
print(valid_chips)

cat("\n=== 分组内有效chip数（样本数）验证 ===\n")
print(group_area_stats %>% select(group, area_m, celltype_unit, n_samples) %>% distinct(), digits = 0)

cat("\n=== Hip 脑区分布统计 ===\n")

In [ ]:
# 加载所需包
library(dplyr)
library(ggplot2)
library(tidyr)
library(ggpubr)

# 1. 读取数据 & 预处理（保留单个样本维度，与上一步一致）
celldata = readRDS('../results/05_nvu_analysis_results/Cortex/200/Cortex_combined_metadata.rds')
areadata = read.csv('../data/Cortex_region_bin50_summary.csv')

# 1.1 脑区合并映射
region_merge_map <- tibble(
  region = c("L1", "L2", "L23", "L3", "L4", "L45", "L456", "L5", "L6", "WM"),
  region_merged = c(
    "L1", "L23", "L23", "L23", "L456", "L456", "L456", "L456", "L456", "WM"
  )
)

# 1.2 筛选血管细胞数据
vascular_cell_types <- c("Endo", "Pericyte", "VSMC")
celldata_vascular <- celldata %>%
  filter(
    # isTrue == 'Matched',
    celltype_unit %in% vascular_cell_types
  ) %>%
  rename(
    region = area,
    sample = sample_id
  ) %>%
  left_join(region_merge_map, by = "region") %>%
  filter(!is.na(region_merged))%>%
  # 将 Pericyte 和 VSMC 合并为 Mural（壁细胞）
  mutate(
    celltype_unit = case_when(
      celltype_unit %in% c("Pericyte", "VSMC") ~ "Mural",
      TRUE ~ celltype_unit
    )
  )

# 1.3 处理areadata：单个样本-合并脑区面积唯一化
sample_group_map <- areadata %>%
  filter(region != "NAN" & !is.na(region)) %>%
  left_join(region_merge_map, by = "region") %>%
  filter(!is.na(region_merged)) %>%
  group_by(sample, group, region_merged) %>%
  summarise(
    sample_region_area_mm2 = sum(area_mm2),
    .groups = "drop"
  ) %>%
  group_by(sample, region_merged) %>%
  mutate(duplicate_flag = n()) %>%
  ungroup() %>%
  filter(duplicate_flag == 1) %>%
  select(-duplicate_flag)

# 1.4 合并数据并保留单个样本维度
celldata_with_group <- celldata_vascular %>%
  left_join(
    sample_group_map,
    by = c("sample", "region_merged")
  ) %>%
  filter(!is.na(group))

# 1.5 单个样本密度统计（基础数据）
sample_region_cell_stats <- celldata_with_group %>%
  group_by(sample, group, region_merged, celltype_unit) %>%
  summarise(
    cell_number = n(),
    sample_region_area = unique(sample_region_area_mm2),
    .groups = "drop"
  ) %>%
  mutate(
    density_per_mm2 = cell_number / sample_region_area
  )
write.csv(sample_region_cell_stats,'../results/Figure1/Cortex_density.csv')

# 2. 关键步骤：计算分组统计量（均值、标准差、标准误）
group_region_stats <- sample_region_cell_stats %>%
  group_by(group, region_merged, celltype_unit) %>%
  summarise(
    # 核心统计量
    mean_density = mean(density_per_mm2),  # 分组均值（柱状图高度）
    sd_density = sd(density_per_mm2),      # 标准差（误差棒：反映离散程度）
    se_density = sd(density_per_mm2) / sqrt(n()),  # 标准误（误差棒：反映均值可靠性）
    n_samples = n(),  # 分组内样本数（验证AD/Con各6个）
    .groups = "drop"
  )

# 3. 可视化：带误差棒的分组对比图（横坐标=region_merged）
# 选择误差棒类型：sd_density（标准差） 或 se_density（标准误），按需切换
error_type <- "se_density"  # 可改为 "se_density"

p_error <- ggplot() +
  # 1. 绘制单个样本数据点（展示个体差异，可选）
  geom_jitter(
    data = sample_region_cell_stats,
    aes(x = region_merged, y = density_per_mm2, color = group),
    position = position_jitterdodge(dodge.width = 0.9, jitter.width = 0.2),
    alpha = 0.6, size = 2
  ) +
  # 2. 绘制分组均值柱状图
  geom_col(
    data = group_region_stats,
    aes(x = region_merged, y = mean_density, fill = group),
    position = "dodge", alpha = 0.8, width = 0.9
  ) +
  # 3. 绘制误差棒（均值±误差，width=误差棒两端横线宽度）
  geom_errorbar(
    data = group_region_stats,
    aes(
      x = region_merged,
      ymin = mean_density - .data[[error_type]],  # 动态调用误差类型
      ymax = mean_density + .data[[error_type]],
      group = group
    ),
    position = position_dodge(width = 0.9),
    width = 0.2, color = "black", linewidth = 0.5
  ) +
  # 4. 按细胞类型分面（清晰展示不同血管细胞差异）
  facet_wrap(~celltype_unit, scales = "free_y") +
  # 5. 标签与主题设置
  labs(
    title = "AD vs Con 各合并脑区血管细胞密度对比（带误差棒）",
    x = "合并脑区",
    y = paste0("细胞密度（个/平方毫米）\n（误差棒：", ifelse(error_type=="sd_density", "标准差", "标准误"), "）"),
    fill = "分组",
    color = "分组"
  ) +
  # 6. 美化主题
  theme_bw() +
  theme(
    plot.title = element_text(hjust = 0.5, size = 14, face = "bold"),
    axis.text.x = element_text(angle = 0, hjust = 0.5, size = 10),
    axis.title = element_text(size = 11),
    legend.position = "top",
    strip.text = element_text(size = 10, face = "bold")
  )

# 4. 展示图形
print(p_error)

# 5. 保存图形（高分辨率，适合论文展示）
ggsave(
  "../results/figure1/AD_Con_density_comparison_with_error_bars.png",
  p_error,
  width = 12, height = 8, dpi = 300
)

# 6. 验证分组样本数（确保AD/Con各6个样本）
print(group_region_stats %>% select(group, region_merged, celltype_unit, n_samples) %>% distinct(), digits = 0)

In [ ]:
areadata = read.csv('../data/Hip_region_bin50_summary.csv')

In [ ]:
# 加载所需包
library(dplyr)
library(ggplot2)
library(tidyr)
library(ggpubr)

# 1. 读取海马体数据
celldata = readRDS('../results/05_nvu_analysis_results/Hip/300/Hip_combined_metadata.rds')
areadata = read.csv('../data/Hip_region_bin50_summary.csv')
areadata$area_m = areadata$area
areadata = areadata[areadata$area_m%in%c('CA1','CA2','CA3','CA4','FAS','SLRM','DG'),]
celldata = celldata[celldata$area_m%in%c('CA1','CA2','CA3','CA4','FAS','SLRM','DG'),]

# 2. 数据预处理
# 2.1 定义血管细胞类型
vascular_cell_types <- c("Endo", "Pericyte")

# 2.2 筛选血管细胞数据 + 关联chip信息
chip_sample_map <- areadata %>%
  distinct(chip, sample) %>%
  filter(!is.na(chip) & !is.na(sample))

celldata_vascular <- celldata %>%
  filter(celltype_unit %in% vascular_cell_types) %>%
  rename(celldata_sample = sample_id) %>%
  filter(!is.na(area_m)) %>%
  left_join(chip_sample_map, by = c("celldata_sample" = "sample")) %>%
  filter(!is.na(chip))%>%
  # 将 Pericyte 和 VSMC 合并为 Mural（壁细胞）
  mutate(
    celltype_unit = case_when(
      celltype_unit %in% c("Pericyte", "VSMC") ~ "Mural",
      TRUE ~ celltype_unit
    )
  )

# 2.3 构建chip-group映射
chip_group_info <- areadata %>%
  distinct(chip) %>%
  mutate(
    group = case_when(
      grepl("AD", chip) ~ "AD",
      grepl("control|Con", chip, ignore.case = TRUE) ~ "Control",
      TRUE ~ "Unknown"
    )
  ) %>%
  filter(group != "Unknown")

# 2.4 合并celldata和分组信息
celldata_with_group <- celldata_vascular %>%
  left_join(chip_group_info, by = "chip") %>%
  filter(!is.na(group))

# 3. 计算相对密度
# 3.1 首先计算每个脑区每种细胞类型的总细胞数
total_cells_per_area <- celldata_with_group %>%
  group_by(area_m, celltype_unit) %>%
  summarise(
    total_cells_in_area = n(),
    .groups = "drop"
  )

# 3.2 计算单个样本（chip层面）的细胞数
sample_cell_counts <- celldata_with_group %>%
  group_by(chip, group, area_m, celltype_unit) %>%
  summarise(
    cell_number = n(),
    .groups = "drop"
  )

# 3.3 合并并计算相对密度
sample_area_cell_stats <- sample_cell_counts %>%
  left_join(total_cells_per_area, by = c("area_m", "celltype_unit")) %>%
  mutate(
    relative_density = cell_number / total_cells_in_area  # 相对密度：该样本细胞数 / 该脑区总细胞数
  )
sample_area_cell_stats = sample_area_cell_stats[!sample_area_cell_stats$chip%in%c('Con3.1','Con3.2'),]
# 4. 计算分组统计量
group_area_stats <- sample_area_cell_stats %>%
  group_by(group, area_m, celltype_unit) %>%
  summarise(
    mean_density = mean(relative_density),
    sd_density = sd(relative_density),
    se_density = sd(relative_density) / sqrt(n()),
    n_samples = n(),
    .groups = "drop"
  )

# 5. 可视化
error_type <- "se_density"

p_hip_error <- ggplot() +
  geom_jitter(
    data = sample_area_cell_stats,
    aes(x = area_m, y = relative_density, color = group),
    position = position_jitterdodge(dodge.width = 0.9, jitter.width = 0.2),
    alpha = 0.6, size = 2
  ) +
  geom_col(
    data = group_area_stats,
    aes(x = area_m, y = mean_density, fill = group),
    position = "dodge", alpha = 0.8, width = 0.9
  ) +
  geom_errorbar(
    data = group_area_stats,
    aes(
      x = area_m,
      ymin = mean_density - .data[[error_type]],
      ymax = mean_density + .data[[error_type]],
      group = group
    ),
    position = position_dodge(width = 0.9),
    width = 0.2, color = "black", linewidth = 0.5
  ) +
  facet_wrap(~celltype_unit, scales = "free_y") +
  labs(
    title = "Hip region AD vs Con relative density",
    x = "（area_m）",
    y = paste0("relative density"),
    fill = "group",
    color = "group"
  ) +
  theme_bw() +
  theme(
    plot.title = element_text(hjust = 0.5, size = 14, face = "bold"),
    axis.text.x = element_text(angle = 45, hjust = 1, size = 10),
    axis.title = element_text(size = 11),
    legend.position = "top",
    strip.text = element_text(size = 10, face = "bold")
  )

# 6. 展示图形并保存
print(p_hip_error)

ggsave(
  "../results/figure1/Hip_AD_Con_relative_density_comparison.png",
  p_hip_error,
  width = 14, height = 8, dpi = 300
)

# 7. 验证结果
print(total_cells_per_area)

cat("\n=== 有效chip清单 ===\n")
print(sample_area_cell_stats %>% distinct(chip, group))

cat("\n=== 分组统计 ===\n")
print(group_area_stats)

In [ ]:
# Vascular Score by Brain Region Analysis
# 顶刊风格绑图

library(tidyverse)
library(ggplot2)
library(ggpubr)
library(ggsignif)
library(cowplot)
library(RColorBrewer)

# 读取数据
hip_Vasular <- read.csv('../results/01_Ficture/Hip/sample_area_m_vascular_stats.csv')
Cortex_Vasular <- read.csv('../results/01_Ficture/Cortex/sample_region_vascular_stats.csv')

# 1. 为Cortex添加group信息
# 根据sample_id添加group
Cortex_Vasular <- Cortex_Vasular %>%
  mutate(group = case_when(
    grepl("B02009F6|B02008D2|C02248B5|A02092E1|B02008C6", sample_id) ~ "AD",
    grepl("B01806B6|B01809A4|D02175A4|D02175A6|B01809A3|B01806B5", sample_id) ~ "Control",
    TRUE ~ NA_character_
  ))

# 检查分组

# 2. Hip数据转换为长数据
hip_areas <- c('CA1', 'CA2', 'CA3', 'CA4', 'DG', 'FAS', 'SLRM')

# 提取mean_score列
hip_mean_long <- hip_Vasular %>%
  select(sample_id, group, ends_with("_mean_score")) %>%
  select(sample_id, group, matches(paste0("^(", paste(hip_areas, collapse="|"), ")_mean_score$"))) %>%
  pivot_longer(
    cols = -c(sample_id, group),
    names_to = "area",
    values_to = "mean_score"
  ) %>%
  mutate(area = gsub("_mean_score", "", area)) %>%
  filter(area %in% hip_areas)

# 提取sum_score列
hip_sum_long <- hip_Vasular %>%
  select(sample_id, group, ends_with("_sum_score")) %>%
  select(sample_id, group, matches(paste0("^(", paste(hip_areas, collapse="|"), ")_sum_score$"))) %>%
  pivot_longer(
    cols = -c(sample_id, group),
    names_to = "area",
    values_to = "sum_score"
  ) %>%
  mutate(area = gsub("_sum_score", "", area)) %>%
  filter(area %in% hip_areas)

# 合并
hip_long <- left_join(hip_mean_long, hip_sum_long, by = c("sample_id", "group", "area"))
hip_long$region <- "Hippocampus"
hip_long = hip_long[!hip_long$sample_id%in%c('D01574C2','D01574C6'),]

# 3. Cortex数据转换为长数据
cortex_areas <- c('L1', 'L23', 'L456', 'WM')

# 提取mean_score列
cortex_mean_long <- Cortex_Vasular %>%
  select(sample_id, group, ends_with("_mean_score")) %>%
  pivot_longer(
    cols = -c(sample_id, group),
    names_to = "area",
    values_to = "mean_score"
  ) %>%
  mutate(area = gsub("_mean_score", "", area)) %>%
  filter(area %in% cortex_areas)

# 提取sum_score列
cortex_sum_long <- Cortex_Vasular %>%
  select(sample_id, group, ends_with("_sum_score")) %>%
  pivot_longer(
    cols = -c(sample_id, group),
    names_to = "area",
    values_to = "sum_score"
  ) %>%
  mutate(area = gsub("_sum_score", "", area)) %>%
  filter(area %in% cortex_areas)

# 合并
cortex_long <- left_join(cortex_mean_long, cortex_sum_long, by = c("sample_id", "group", "area"))
cortex_long$region <- "Cortex"


# 4. 顶刊风格主题设置
theme_publication <- function(base_size = 12) {
  theme_classic(base_size = base_size) +
    theme(
      # 标题
      plot.title = element_text(face = "bold", size = rel(1.2), hjust = 0.5),
      # 坐标轴
      axis.title = element_text(face = "bold", size = rel(1)),
      axis.title.y = element_text(angle = 90, vjust = 2),
      axis.title.x = element_text(vjust = -0.2),
      axis.text = element_text(size = rel(0.9), color = "black"),
      axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1),
      axis.line = element_line(colour = "black", linewidth = 0.5),
      axis.ticks = element_line(colour = "black", linewidth = 0.5),
      # 图例
      legend.title = element_text(face = "bold"),
      legend.position = "top",
      legend.direction = "horizontal",
      legend.key.size = unit(0.5, "cm"),
      legend.background = element_blank(),
      # 面板
      panel.background = element_blank(),
      panel.grid.major = element_blank(),
      panel.grid.minor = element_blank(),
      panel.border = element_blank(),
      # 边距
      plot.margin = margin(10, 10, 10, 10)
    )
}

# 配色方案
colors_group <- c("AD" = "#E64B35", "Control" = "#4DBBD5")

# 5. Hippocampus绑图
# 设置area顺序
hip_long$area <- factor(hip_long$area, levels = c('CA1', 'CA2', 'CA3', 'CA4', 'DG', 'FAS', 'SLRM'))
hip_long <- hip_long %>% filter(!is.na(group))

# Mean Score绑图
p_hip_mean <- ggplot(hip_long, aes(x = area, y = mean_score, fill = group)) +
  geom_boxplot(outlier.shape = NA, alpha = 0.7, width = 0.6, position = position_dodge(0.75)) +
  geom_point(aes(color = group), position = position_jitterdodge(jitter.width = 0.1, dodge.width = 0.75),
             size = 1.5, alpha = 0.8) +
  scale_fill_manual(values = colors_group) +
  scale_color_manual(values = colors_group) +
  labs(
    title = "Hippocampus - Mean Vascular Score",
    x = "Brain Region",
    y = "Mean Vascular Score",
    fill = "Group",
    color = "Group"
  ) +
  theme_publication() +
  stat_compare_means(aes(group = group), method = "wilcox.test", 
                     label = "p.signif", hide.ns = TRUE,
                     label.y.npc = 0.95)

# Sum Score绑图
p_hip_sum <- ggplot(hip_long, aes(x = area, y = sum_score, fill = group)) +
  geom_boxplot(outlier.shape = NA, alpha = 0.7, width = 0.6, position = position_dodge(0.75)) +
  geom_point(aes(color = group), position = position_jitterdodge(jitter.width = 0.1, dodge.width = 0.75),
             size = 1.5, alpha = 0.8) +
  scale_fill_manual(values = colors_group) +
  scale_color_manual(values = colors_group) +
  labs(
    title = "Hippocampus - Sum Vascular Score",
    x = "Brain Region",
    y = "Sum Vascular Score",
    fill = "Group",
    color = "Group"
  ) +
  theme_publication() +
  stat_compare_means(aes(group = group), method = "wilcox.test", 
                     label = "p.signif", hide.ns = TRUE,
                     label.y.npc = 0.95)

# 6. Cortex绑图
# 设置area顺序
cortex_long$area <- factor(cortex_long$area, levels = c('L1', 'L23', 'L456', 'WM'))
cortex_long <- cortex_long %>% filter(!is.na(group))

# Mean Score绑图
p_cortex_mean <- ggplot(cortex_long, aes(x = area, y = mean_score, fill = group)) +
  geom_boxplot(outlier.shape = NA, alpha = 0.7, width = 0.6, position = position_dodge(0.75)) +
  geom_point(aes(color = group), position = position_jitterdodge(jitter.width = 0.1, dodge.width = 0.75),
             size = 1.5, alpha = 0.8) +
  scale_fill_manual(values = colors_group) +
  scale_color_manual(values = colors_group) +
  labs(
    title = "Cortex - Mean Vascular Score",
    x = "Cortical Layer",
    y = "Mean Vascular Score",
    fill = "Group",
    color = "Group"
  ) +
  theme_publication() +
  stat_compare_means(aes(group = group), method = "wilcox.test", 
                     label = "p.signif", hide.ns = TRUE,
                     label.y.npc = 0.95)

# Sum Score绑图
p_cortex_sum <- ggplot(cortex_long, aes(x = area, y = sum_score, fill = group)) +
  geom_boxplot(outlier.shape = NA, alpha = 0.7, width = 0.6, position = position_dodge(0.75)) +
  geom_point(aes(color = group), position = position_jitterdodge(jitter.width = 0.1, dodge.width = 0.75),
             size = 1.5, alpha = 0.8) +
  scale_fill_manual(values = colors_group) +
  scale_color_manual(values = colors_group) +
  labs(
    title = "Cortex - Sum Vascular Score",
    x = "Cortical Layer",
    y = "Sum Vascular Score",
    fill = "Group",
    color = "Group"
  ) +
  theme_publication() +
  stat_compare_means(aes(group = group), method = "wilcox.test", 
                     label = "p.signif", hide.ns = TRUE,
                     label.y.npc = 0.95)

# 7. 组合图
# Mean Score组合图
p_mean_combined <- plot_grid(
  p_hip_mean, p_cortex_mean,
  labels = c("A", "B"),
  ncol = 2,
  align = "h",
  rel_widths = c(1.2, 1)
)

# Sum Score组合图
p_sum_combined <- plot_grid(
  p_hip_sum, p_cortex_sum,
  labels = c("A", "B"),
  ncol = 2,
  align = "h",
  rel_widths = c(1.2, 1)
)

# 全部组合
p_all <- plot_grid(
  p_mean_combined,
  p_sum_combined,
  labels = c("", ""),
  ncol = 1,
  rel_heights = c(1, 1)
)

# 8. 保存图片
# 单独保存
ggsave("../results/Figure1/Hip_mean_vascular_score.pdf", p_hip_mean, width = 8, height = 6, dpi = 300)
ggsave("../results/Figure1/Hip_sum_vascular_score.pdf", p_hip_sum, width = 8, height = 6, dpi = 300)
ggsave("../results/Figure1/Cortex_mean_vascular_score.pdf", p_cortex_mean, width = 6, height = 6, dpi = 300)
ggsave("../results/Figure1/Cortex_sum_vascular_score.pdf", p_cortex_sum, width = 6, height = 6, dpi = 300)

# 组合图保存
ggsave("../results/Figure1/Vascular_mean_score_combined.pdf", p_mean_combined, width = 14, height = 6, dpi = 300)
ggsave("../results/Figure1/Vascular_sum_score_combined.pdf", p_sum_combined, width = 14, height = 6, dpi = 300)
ggsave("../results/Figure1/Vascular_score_all.pdf", p_all, width = 14, height = 12, dpi = 300)

# PNG格式

print("All plots saved successfully!")

# 9. 保存长数据用于后续分析
write.csv(hip_long, "../results/Figure1/hip_vascular_long.csv", row.names = FALSE)
write.csv(cortex_long, "../results/Figure1/cortex_vascular_long.csv", row.names = FALSE)
print("Long format data saved!")

In [ ]:
# 加载所需包
library(dplyr)
library(ggplot2)
library(tidyr)
library(ggpubr)
library(cowplot)

# 1. 读取海马体数据
celldata = readRDS('../results/05_nvu_analysis_results/Hip/300/Hip_combined_metadata.rds')
areadata = read.csv('../data/Hip_region_bin50_summary.csv')
areadata$area_m = areadata$area
areadata = areadata[areadata$area_m %in% c('CA1','CA2','CA3','CA4','FAS','SLRM','DG'),]
celldata = celldata[celldata$area_m %in% c('CA1','CA2','CA3','CA4','FAS','SLRM','DG'),]

# Vascular score数据
Hip_vascular <- read.csv('../results/01_Ficture/Hip/sample_area_m_vascular_stats.csv')

# 2. 数据预处理
# 2.1 定义血管细胞类型并合并壁细胞
vascular_cell_types <- c("Endo", "Pericyte")

# 2.2 筛选血管细胞数据 + 关联chip信息
chip_sample_map <- areadata %>%
  distinct(chip, sample) %>%
  filter(!is.na(chip) & !is.na(sample))

celldata_vascular <- celldata %>%
  filter(celltype_unit %in% vascular_cell_types) %>%
  rename(celldata_sample = sample_id) %>%
  filter(!is.na(area_m)) %>%
  left_join(chip_sample_map, by = c("celldata_sample" = "sample")) %>%
  filter(!is.na(chip)) %>%
  # 将 Pericyte 和 VSMC 合并为 Mural（壁细胞）
  mutate(
    celltype_unit = case_when(
      celltype_unit %in% c("Pericyte") ~ "Mural",
      TRUE ~ celltype_unit
    )
  )

# 2.3 构建chip-group映射
chip_group_info <- areadata %>%
  distinct(chip) %>%
  mutate(
    group = case_when(
      grepl("AD", chip) ~ "AD",
      grepl("control|Con", chip, ignore.case = TRUE) ~ "Control",
      TRUE ~ "Unknown"
    )
  ) %>%
  filter(group != "Unknown")

# 2.4 合并celldata和分组信息
celldata_with_group <- celldata_vascular %>%
  left_join(chip_group_info, by = "chip") %>%
  filter(!is.na(group))

# 3. 计算相对密度
# 3.1 计算每个脑区每种细胞类型的总细胞数
total_cells_per_area <- celldata_with_group %>%
  group_by(area_m, celltype_unit) %>%
  summarise(
    total_cells_in_area = n(),
    .groups = "drop"
  )

print(total_cells_per_area)

# 3.2 计算单个样本（chip层面）的细胞数
sample_cell_counts <- celldata_with_group %>%
  group_by(chip, celldata_sample, group, area_m, celltype_unit) %>%
  summarise(
    cell_number = n(),
    .groups = "drop"
  )

# 3.3 合并并计算相对密度
sample_area_cell_stats <- sample_cell_counts %>%
  left_join(total_cells_per_area, by = c("area_m", "celltype_unit")) %>%
  mutate(
    relative_density = cell_number / total_cells_in_area
  )

# 排除特定样本
sample_area_cell_stats <- sample_area_cell_stats %>%
  filter(!chip %in% c('Con3.1', 'Con3.2'))


# 4. 处理Vascular Score数据
hip_areas <- c('CA1', 'CA2', 'CA3', 'CA4', 'FAS', 'SLRM', 'DG')

# 提取mean_score并转换为长格式
hip_mean_long <- Hip_vascular %>%
  select(sample_id, ends_with("_mean_score")) %>%
  pivot_longer(
    cols = -sample_id,
    names_to = "area",
    values_to = "mean_score"
  ) %>%
  mutate(area = gsub("_mean_score", "", area)) %>%
  filter(area %in% hip_areas) %>%
  rename(area_m = area, sample = sample_id)

print("Hip mean_score long format:")

# 检查sample名称格式

# 5. 合并相对密度数据与mean_score
combined_data <- sample_area_cell_stats %>%
  left_join(
    hip_mean_long %>% select(sample, area_m, mean_score),
    by = c("celldata_sample" = "sample", "area_m")
  ) %>%
  filter(!is.na(mean_score))

# 计算新score: relative_density × mean_score^2
combined_data <- combined_data %>%
  mutate(
    relative_density_x_meanscore = relative_density * (mean_score) * (mean_score)
  )


# 6. 顶刊风格主题
theme_publication <- function(base_size = 12) {
  theme_classic(base_size = base_size) +
    theme(
      plot.title = element_text(face = "bold", size = rel(1.2), hjust = 0.5),
      axis.title = element_text(face = "bold", size = rel(1)),
      axis.title.y = element_text(angle = 90, vjust = 2),
      axis.title.x = element_text(vjust = -0.2),
      axis.text = element_text(size = rel(0.9), color = "black"),
      axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1),
      axis.line = element_line(colour = "black", linewidth = 0.5),
      axis.ticks = element_line(colour = "black", linewidth = 0.5),
      legend.title = element_text(face = "bold"),
      legend.position = "top",
      legend.direction = "horizontal",
      legend.background = element_blank(),
      panel.background = element_blank(),
      panel.grid.major = element_blank(),
      panel.grid.minor = element_blank(),
      strip.background = element_blank(),
      strip.text = element_text(face = "bold", size = rel(1))
    )
}

colors_group <- c("AD" = "#E64B35", "Control" = "#4DBBD5")

# 设置脑区顺序
combined_data$area_m <- factor(combined_data$area_m, 
                               levels = c('CA1', 'CA2', 'CA3', 'CA4', 'DG', 'FAS', 'SLRM'))


# 7.2 各细胞类型单独绑图
plot_celltype <- function(data, celltype, colors) {
  df <- data %>% filter(celltype_unit == celltype)
  
  p <- ggplot(df, aes(x = area_m, y = relative_density_x_meanscore, fill = group)) +
    geom_boxplot(outlier.shape = NA, alpha = 0.7, width = 0.6, position = position_dodge(0.75)) +
    geom_point(aes(color = group), 
               position = position_jitterdodge(jitter.width = 0.1, dodge.width = 0.75),
               size = 2.5, alpha = 0.8) +
    scale_fill_manual(values = colors) +
    scale_color_manual(values = colors) +
    labs(
      title = paste0(celltype, " - Relative Density × Mean Score²"),
      x = "Hippocampal Region",
      y = "Relative Density × Mean Score²",
      fill = "Group",
      color = "Group"
    ) +
    theme_publication() +
    stat_compare_means(aes(group = group), method = "wilcox.test", 
                       label = "p.signif", hide.ns = FALSE,
                       label.y.npc = 0.95, size = 4)
  return(p)
}

p_endo <- plot_celltype(combined_data, "Endo", colors_group)
p_Mural <- plot_celltype(combined_data, "Mural", colors_group)

# 组合图
p_combined <- plot_grid(
  p_endo, p_Mural,
  labels = c("A", "B"),
  ncol = 2,
  align = "h"
)

ggsave('../results/Figure1/Hip_relative_density_x_meanscore_combined.pdf', 
       p_combined, width = 14, height = 6, dpi = 300)
ggsave('../results/Figure1/Hip_relative_density_x_meanscore_combined.png', 
       p_combined, width = 14, height = 6, dpi = 300)

# 单独保存
ggsave('../results/Figure1/Hip_Endo_relative_density_x_meanscore.pdf', 
       p_endo, width = 8, height = 6, dpi = 300)
ggsave('../results/Figure1/Hip_Pericyte_relative_density_x_meanscore.pdf', 
       p_pericyte, width = 8, height = 6, dpi = 300)

# 8. 汇总统计表
summary_stats <- combined_data %>%
  group_by(area_m, celltype_unit, group) %>%
  summarise(
    n = n(),
    mean_relative_density = mean(relative_density, na.rm = TRUE),
    sd_relative_density = sd(relative_density, na.rm = TRUE),
    mean_vascular_score = mean(mean_score, na.rm = TRUE),
    sd_vascular_score = sd(mean_score, na.rm = TRUE),
    mean_relative_density_x_score = mean(relative_density_x_meanscore, na.rm = TRUE),
    sd_relative_density_x_score = sd(relative_density_x_meanscore, na.rm = TRUE),
    .groups = "drop"
  )

write.csv(summary_stats, 
          '../results/Figure1/Hip_relative_density_x_meanscore_summary.csv',
          row.names = FALSE)

# 保存完整数据
write.csv(combined_data,
          '../results/Figure1/Hip_relative_density_x_meanscore_data.csv',
          row.names = FALSE)

print(summary_stats)

print("\n========================================")
print("All outputs saved successfully!")

In [ ]:
# Relative Density × Mean Score Analysis
# 计算血管细胞相对密度与vascular score的乘积

library(tidyverse)
library(ggpubr)
library(cowplot)

# 1. 读取数据
# 细胞密度数据
celldata <- readRDS('../results/05_nvu_analysis_results/Cortex/300/Cortex_combined_metadata.rds')
areadata <- read.csv('../data/Cortex_region_bin50_summary.csv')

# Vascular score数据
cortex_vascular <- read.csv('../results/01_Ficture/Hip/sample_area_m_vascular_stats.csv')


# 筛选血管细胞数据
vascular_cell_types <- c("Endo", "Pericyte", "VSMC")
celldata_vascular <- celldata %>%
  filter(celltype_unit %in% vascular_cell_types) %>%
  rename(region = area, sample = sample_id) %>%
  left_join(region_merge_map, by = "region") %>%
  filter(!is.na(region_merged)) %>%
  # 将 Pericyte 和 VSMC 合并为 Mural（壁细胞）
  mutate(
    celltype_unit = case_when(
      celltype_unit %in% c("Pericyte", "VSMC") ~ "Mural",
      TRUE ~ celltype_unit
    )
  )

# 处理areadata - 获取分组信息
sample_group_map <- areadata %>%
  filter(region != "NAN" & !is.na(region)) %>%
  distinct(sample, group)

# 合并分组信息
celldata_with_group <- celldata_vascular %>%
  left_join(sample_group_map, by = "sample") %>%
  filter(!is.na(group))

# 3. 计算相对密度
# 3.1 计算每个脑区每种细胞类型的总细胞数
total_cells_per_region <- celldata_with_group %>%
  group_by(region_merged, celltype_unit) %>%
  summarise(
    total_cells_in_region = n(),
    .groups = "drop"
  )

print(total_cells_per_region)

# 3.2 计算单个样本的细胞数
sample_cell_counts <- celldata_with_group %>%
  group_by(sample, group, region_merged, celltype_unit) %>%
  summarise(
    cell_number = n(),
    .groups = "drop"
  )

# 3.3 合并并计算相对密度
sample_region_cell_stats <- sample_cell_counts %>%
  left_join(total_cells_per_region, by = c("region_merged", "celltype_unit")) %>%
  mutate(
    relative_density = cell_number / total_cells_in_region  # 相对密度
  )


# 4. 处理Vascular Score数据 - 转换为长格式
cortex_areas <- c('L1', 'L23', 'L456', 'WM')

# 提取mean_score
cortex_mean_long <- cortex_vascular %>%
  select(sample_id, ends_with("_mean_score")) %>%
  pivot_longer(
    cols = -sample_id,
    names_to = "area",
    values_to = "mean_score"
  ) %>%
  mutate(area = gsub("_mean_score", "", area)) %>%
  filter(area %in% cortex_areas) %>%
  rename(region_merged = area, sample = sample_id)

print("Cortex mean_score long format:")

# 5. 合并相对密度数据与mean_score
# 检查sample名称格式

# 合并数据
combined_data <- sample_region_cell_stats %>%
  left_join(
    cortex_mean_long %>% select(sample, region_merged, mean_score),
    by = c("sample", "region_merged")
  ) %>%
  filter(!is.na(mean_score))

# 计算新score: relative_density × mean_score^2
combined_data <- combined_data %>%
  mutate(
    relative_density_x_meanscore = relative_density * (mean_score) * (mean_score)
  )


# 6. 顶刊风格主题
theme_publication <- function(base_size = 12) {
  theme_classic(base_size = base_size) +
    theme(
      plot.title = element_text(face = "bold", size = rel(1.2), hjust = 0.5),
      axis.title = element_text(face = "bold", size = rel(1)),
      axis.title.y = element_text(angle = 90, vjust = 2),
      axis.title.x = element_text(vjust = -0.2),
      axis.text = element_text(size = rel(0.9), color = "black"),
      axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1),
      axis.line = element_line(colour = "black", linewidth = 0.5),
      axis.ticks = element_line(colour = "black", linewidth = 0.5),
      legend.title = element_text(face = "bold"),
      legend.position = "top",
      legend.direction = "horizontal",
      legend.background = element_blank(),
      panel.background = element_blank(),
      panel.grid.major = element_blank(),
      panel.grid.minor = element_blank(),
      strip.background = element_blank(),
      strip.text = element_text(face = "bold", size = rel(1))
    )
}

colors_group <- c("AD" = "#E64B35", "Control" = "#4DBBD5")

# 设置脑区顺序
combined_data$region_merged <- factor(combined_data$region_merged, 
                                       levels = c('L1', 'L23', 'L456', 'WM'))

# 7. 绑图 - 按细胞类型分面

# 7.1 Relative Density × Mean Score 箱线图（按细胞类型分面）
p_facet <- ggplot(combined_data, aes(x = region_merged, y = relative_density_x_meanscore, fill = group)) +
  geom_boxplot(outlier.shape = NA, alpha = 0.7, width = 0.6, position = position_dodge(0.75)) +
  geom_point(aes(color = group), 
             position = position_jitterdodge(jitter.width = 0.1, dodge.width = 0.75),
             size = 2, alpha = 0.8) +
  facet_wrap(~celltype_unit, scales = "free_y", ncol = 3) +
  scale_fill_manual(values = colors_group) +
  scale_color_manual(values = colors_group) +
  labs(
    title = "Cortex - Vascular Cell Relative Density × Mean Score²",
    x = "Cortical Layer",
    y = "Relative Density × Mean Score²",
    fill = "Group",
    color = "Group"
  ) +
  theme_publication() +
  stat_compare_means(aes(group = group), method = "wilcox.test", 
                     label = "p.signif", hide.ns = FALSE,
                     label.y.npc = 0.95, size = 3.5)

ggsave('../results/Figure1/Cortex_relative_density_x_meanscore_facet.pdf', 
       p_facet, width = 12, height = 5, dpi = 300)
ggsave('../results/Figure1/Cortex_relative_density_x_meanscore_facet.png', 
       p_facet, width = 12, height = 5, dpi = 300)

# 7.2 各细胞类型单独绑图
plot_celltype <- function(data, celltype, colors) {
  df <- data %>% filter(celltype_unit == celltype)
  
  p <- ggplot(df, aes(x = region_merged, y = relative_density_x_meanscore, fill = group)) +
    geom_boxplot(outlier.shape = NA, alpha = 0.7, width = 0.6, position = position_dodge(0.75)) +
    geom_point(aes(color = group), 
               position = position_jitterdodge(jitter.width = 0.1, dodge.width = 0.75),
               size = 2.5, alpha = 0.8) +
    scale_fill_manual(values = colors) +
    scale_color_manual(values = colors) +
    labs(
      title = paste0(celltype, " - Relative Density × Mean Score²"),
      x = "Cortical Layer",
      y = "Relative Density × Mean Score²",
      fill = "Group",
      color = "Group"
    ) +
    theme_publication() +
    stat_compare_means(aes(group = group), method = "wilcox.test", 
                       label = "p.signif", hide.ns = FALSE,
                       label.y.npc = 0.95, size = 4)
  return(p)
}

# 7.2 各细胞类型单独绑图（只需要两个）
p_endo <- plot_celltype(combined_data, "Endo", colors_group)
p_mural <- plot_celltype(combined_data, "Mural", colors_group)

# 组合图（改为两列）
p_combined <- plot_grid(
  p_endo, p_mural,
  labels = c("A", "B"),
  ncol = 2,
  align = "h"
)

# 保存时也相应修改
ggsave('../results/01_Ficture/Cortex/Cortex_relative_density_x_meanscore_combined.pdf', 
       p_combined, width = 10, height = 5, dpi = 300)

# 单独保存
ggsave('../results/01_Ficture/Cortex/Cortex_Endo_relative_density_x_meanscore.pdf', 
       p_endo, width = 6, height = 5, dpi = 300)
ggsave('../results/01_Ficture/Cortex/Cortex_Mural_relative_density_x_meanscore.pdf', 
       p_mural, width = 6, height = 5, dpi = 300)

# 8. 汇总统计表
summary_stats <- combined_data %>%
  group_by(region_merged, celltype_unit, group) %>%
  summarise(
    n = n(),
    mean_relative_density = mean(relative_density, na.rm = TRUE),
    sd_relative_density = sd(relative_density, na.rm = TRUE),
    mean_vascular_score = mean(mean_score, na.rm = TRUE),
    sd_vascular_score = sd(mean_score, na.rm = TRUE),
    mean_relative_density_x_score = mean(relative_density_x_meanscore, na.rm = TRUE),
    sd_relative_density_x_score = sd(relative_density_x_meanscore, na.rm = TRUE),
    .groups = "drop"
  )

write.csv(summary_stats, 
          '../results/Figure1/Cortex_relative_density_x_meanscore_summary.csv',
          row.names = FALSE)

# 保存完整数据
write.csv(combined_data,
          '../results/Figure1/Cortex_relative_density_x_meanscore_data.csv',
          row.names = FALSE)

print(summary_stats)

print("\n========================================")
print("All outputs saved successfully!")

## Density and endothelial/mural score dot plots

In [ ]:
# Vascular Score和Density的Dotplot可视化
# 点大小 = 密度，颜色 = AD与Control的血管分数差值
# 统一legend范围

library(tidyverse)
library(ggplot2)
library(cowplot)

# 1. 读取数据
hip_long <- read.csv("../results/01_Ficture/Hip/Hip_all_samples_combined.csv")
cortex_long <- read.csv("../results/01_Ficture/Cortex/Cortex_all_samples_combined.csv")
hip_density <- read.csv('../results/Figure1/Hip_density.csv')
cortex_density <- read.csv('../results/Figure1/Cortex_density.csv')

hip_density <- hip_density[hip_density$area_m %in% c('CA1','CA2','CA3','CA4','FAS','SLRM','DG'),]
hip_long <- hip_long[hip_long$area_m %in% c('CA1','CA2','CA3','CA4','FAS','SLRM','DG'),]
hip_long <- hip_long[!hip_long$sample_id %in% c('D01574C2','D01574C6'),]
# 2. 准备Hip数据
hip_scores <- hip_long %>%
  group_by(sample_id, group, area_m) %>%
  summarise(
    Endothelial_score = mean(Endothelial_score, na.rm = TRUE),
    Pericyte_score = mean(Pericyte_score, na.rm = TRUE),
    SMC_score = mean(SMC_score, na.rm = TRUE),
    .groups = 'drop'
  ) %>%
  mutate(Mural_score = (Endothelial_score + Pericyte_score) / 2)

hip_scores_summary <- hip_scores %>%
  group_by(area_m, group) %>%
  summarise(
    Endo_mean = mean(Endothelial_score, na.rm = TRUE),
    Mural_mean = mean(Mural_score, na.rm = TRUE),
    .groups = 'drop'
  ) %>%
  pivot_wider(
    names_from = group,
    values_from = c(Endo_mean, Mural_mean)
  ) %>%
  mutate(
    Endo_diff = Endo_mean_AD - Endo_mean_Control,
    Mural_diff = Mural_mean_AD - Mural_mean_Control
  )

hip_density_summary <- hip_density %>%
  group_by(area_m, celltype_unit) %>%
  summarise(
    mean_density = mean(density_per_mm2, na.rm = TRUE),
    .groups = 'drop'
  )

hip_plot_data <- hip_scores_summary %>%
  left_join(
    hip_density_summary %>% filter(celltype_unit == "Endo") %>% 
      select(area_m, Endo_density = mean_density),
    by = "area_m"
  ) %>%
  left_join(
    hip_density_summary %>% filter(celltype_unit == "Mural") %>% 
      select(area_m, Mural_density = mean_density),
    by = "area_m"
  ) %>%
  pivot_longer(
    cols = c(Endo_diff, Mural_diff),
    names_to = "cell_type",
    values_to = "score_diff"
  ) %>%
  mutate(
    density = ifelse(cell_type == "Endo_diff", Endo_density, Mural_density),
    cell_type = gsub("_diff", "", cell_type)
  )

# 3. 准备Cortex数据
cortex_scores <- cortex_long %>%
  group_by(sample_id, group, region_merged) %>%
  summarise(
    Endothelial_score = mean(Endothelial_score, na.rm = TRUE),
    Pericyte_score = mean(Pericyte_score, na.rm = TRUE),
    SMC_score = mean(SMC_score, na.rm = TRUE),
    .groups = 'drop'
  ) %>%
  mutate(Mural_score = (Endothelial_score + SMC_score) / 3)

cortex_scores_summary <- cortex_scores %>%
  group_by(region_merged, group) %>%
  summarise(
    Endo_mean = mean(Endothelial_score, na.rm = TRUE),
    Mural_mean = mean(Mural_score, na.rm = TRUE),
    .groups = 'drop'
  ) %>%
  pivot_wider(
    names_from = group,
    values_from = c(Endo_mean, Mural_mean)
  ) %>%
  mutate(
    Endo_diff = Endo_mean_AD - Endo_mean_Control,
    Mural_diff = Mural_mean_AD - Mural_mean_Control
  )

cortex_density_summary <- cortex_density %>%
  group_by(region_merged, celltype_unit) %>%
  summarise(
    mean_density = mean(density_per_mm2, na.rm = TRUE),
    .groups = 'drop'
  )

cortex_plot_data <- cortex_scores_summary %>%
  left_join(
    cortex_density_summary %>% filter(celltype_unit == "Endo") %>% 
      select(region_merged, Endo_density = mean_density),
    by = "region_merged"
  ) %>%
  left_join(
    cortex_density_summary %>% filter(celltype_unit == "Mural") %>% 
      select(region_merged, Mural_density = mean_density),
    by = "region_merged"
  ) %>%
  pivot_longer(
    cols = c(Endo_diff, Mural_diff),
    names_to = "cell_type",
    values_to = "score_diff"
  ) %>%
  mutate(
    density = ifelse(cell_type == "Endo_diff", Endo_density, Mural_density),
    cell_type = gsub("_diff", "", cell_type)
  )

# 4. 计算统一的legend范围
# 合并两个数据集以计算全局范围
combined_data <- bind_rows(
  hip_plot_data %>% mutate(region = "Hippocampus"),
  cortex_plot_data %>% 
    rename(area_m = region_merged) %>% 
    mutate(region = "Cortex")
)

# 计算统一的密度范围
density_min <- min(combined_data$density, na.rm = TRUE)
density_max <- max(combined_data$density, na.rm = TRUE)
density_breaks <- seq(
  round(density_min), 
  round(density_max), 
  length.out = 4
) %>% round()

# 计算统一的score差值范围（对称）
score_diff_max <- max(abs(combined_data$score_diff), na.rm = TRUE)
score_diff_limits <- c(-score_diff_max, score_diff_max)

print(paste0("Density范围: ", round(density_min, 2), " - ", round(density_max, 2)))
print(paste0("Score差值范围: ", round(score_diff_limits[1], 4), " - ", round(score_diff_limits[2], 4)))

# 5. 主题设置
theme_dotplot <- function(base_size = 12) {
  theme_classic(base_size = base_size) +
    theme(
      plot.title = element_text(face = "bold", size = rel(1.2), hjust = 0.5),
      axis.title = element_text(face = "bold", size = rel(1)),
      axis.text = element_text(size = rel(0.9), color = "black"),
      axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1),
      axis.line = element_line(colour = "black", linewidth = 0.5),
      axis.ticks = element_line(colour = "black", linewidth = 0.5),
      legend.title = element_text(face = "bold", size = rel(0.9)),
      legend.text = element_text(size = rel(0.8)),
      legend.position = "right",
      panel.background = element_blank(),
      panel.grid.major = element_line(color = "grey90", linewidth = 0.2),
      panel.border = element_blank(),
      strip.background = element_rect(fill = "grey95", color = "black"),
      strip.text = element_text(face = "bold")
    )
}

# 6. 绘制Hippocampus Dotplot（统一legend）
hip_areas <- c('CA1', 'CA2', 'CA3', 'CA4', 'DG', 'FAS', 'SLRM')
hip_plot_data$area_m <- factor(hip_plot_data$area_m, levels = hip_areas)
hip_plot_data$cell_type <- factor(hip_plot_data$cell_type, levels = c("Endo", "Mural"))

p_hip <- ggplot(hip_plot_data, aes(x = area_m, y = cell_type)) +
  geom_point(aes(size = density, color = score_diff), alpha = 0.8) +
  scale_size_continuous(
    name = "Density\n(cells/mm²)",
    range = c(2, 15),
    limits = c(density_min, density_max),
    breaks = density_breaks
  ) +
  scale_color_gradient2(
    name = "Score Difference\n(AD - Control)",
    low = "#4575B4",
    mid = "white",
    high = "#D73027",
    midpoint = 0,
    limits = score_diff_limits
  ) +
  labs(
    title = "Hippocampus - Vascular Score & Density",
    x = "Brain Region",
    y = "Cell Type"
  ) +
  theme_dotplot() +
  guides(
    size = guide_legend(order = 1),
    color = guide_colorbar(order = 2)
  )

# 7. 绘制Cortex Dotplot（统一legend）
cortex_regions <- c('L1', 'L23', 'L456', 'WM')
cortex_plot_data$region_merged <- factor(cortex_plot_data$region_merged, levels = cortex_regions)
cortex_plot_data$cell_type <- factor(cortex_plot_data$cell_type, levels = c("Endo", "Mural"))

p_cortex <- ggplot(cortex_plot_data, aes(x = region_merged, y = cell_type)) +
  geom_point(aes(size = density, color = score_diff), alpha = 0.8) +
  scale_size_continuous(
    name = "Density\n(cells/mm²)",
    range = c(2, 15),
    limits = c(density_min, density_max),
    breaks = density_breaks
  ) +
  scale_color_gradient2(
    name = "Score Difference\n(AD - Control)",
    low = "#4575B4",
    mid = "white",
    high = "#D73027",
    midpoint = 0,
    limits = score_diff_limits
  ) +
  labs(
    title = "Cortex - Vascular Score & Density",
    x = "Cortical Layer",
    y = "Cell Type"
  ) +
  theme_dotplot() +
  guides(
    size = guide_legend(order = 1),
    color = guide_colorbar(order = 2)
  )

# 8. 组合图（共享legend）
# 提取legend
legend <- get_legend(
  p_hip + 
    theme(legend.box.margin = margin(0, 0, 0, 12))
)

# 移除各自的legend
p_hip_no_legend <- p_hip + theme(legend.position = "none")
p_cortex_no_legend <- p_cortex + theme(legend.position = "none")

# 组合图：两个图并排，右侧共享legend
p_combined_horizontal <- plot_grid(
  p_hip_no_legend, 
  p_cortex_no_legend,
  legend,
  labels = c("A", "B", ""),
  ncol = 3,
  align = "h",
  rel_widths = c(1.2, 1, 0.4)
)

# 组合图：两个图上下排列，右侧共享legend
p_combined_vertical <- plot_grid(
  plot_grid(
    p_hip_no_legend, 
    p_cortex_no_legend,
    labels = c("A", "B"),
    ncol = 1,
    align = "v"
  ),
  legend,
  ncol = 2,
  rel_widths = c(1, 0.3)
)

# 9. 保存图片
output_dir <- "../results/Figure1"

# 单独保存（带各自legend）
ggsave(file.path(output_dir, "vascular_dotplot_hip.pdf"), p_hip, 
       width = 8, height = 4, dpi = 300)
ggsave(file.path(output_dir, "vascular_dotplot_cortex.pdf"), p_cortex, 
       width = 6, height = 4, dpi = 300)

# 组合图（共享legend - 水平排列）
ggsave(file.path(output_dir, "vascular_dotplot_combined_horizontal.pdf"), 
       p_combined_horizontal, width = 14, height = 5, dpi = 300)

# 组合图（共享legend - 垂直排列）
ggsave(file.path(output_dir, "vascular_dotplot_combined_vertical.pdf"), 
       p_combined_vertical, width = 10, height = 8, dpi = 300)

# 保存数据
write.csv(hip_plot_data, file.path(output_dir, "hip_dotplot_data.csv"), row.names = FALSE)
write.csv(cortex_plot_data, file.path(output_dir, "cortex_dotplot_data.csv"), row.names = FALSE)
write.csv(combined_data, file.path(output_dir, "combined_dotplot_data.csv"), row.names = FALSE)

## Joint vascular-density dot plots

In [ ]:
library(tidyverse)
library(ggplot2)
library(cowplot)

# 1. 读取数据
hip_long <- read.csv("../results/01_Ficture/Hip/Hip_all_samples_combined.csv")
cortex_long <- read.csv("../results/01_Ficture/Cortex/Cortex_all_samples_combined.csv")
hip_density <- read.csv('../results/Figure1/Hip_density.csv')
cortex_density <- read.csv('../results/Figure1/Cortex_density.csv')

hip_density <- hip_density[hip_density$area_m %in% c('CA1','CA2','CA3','CA4','FAS','SLRM','DG'),]
hip_long <- hip_long[hip_long$area_m %in% c('CA1','CA2','CA3','CA4','FAS','SLRM','DG'),]
hip_long <- hip_long[!hip_long$sample_id %in% c('D01574C2','D01574C6'),]

# 2. 准备Hip Vascular Score数据（使用score_norm的sum）
hip_scores <- hip_long %>%
  group_by(sample_id, group, area_m) %>%
  summarise(
    Endothelial_score = sum(Endothelial_score_norm, na.rm = TRUE),  # 改为sum
    Pericyte_score = sum(Pericyte_score_norm, na.rm = TRUE),        # 改为sum
    SMC_score = sum(SMC_score_norm, na.rm = TRUE),                  # 改为sum
    .groups = 'drop'
  ) %>%
  mutate(Mural_score = Endothelial_score + Pericyte_score)  # Mural是两者之和

hip_scores_summary <- hip_scores %>%
  group_by(area_m, group) %>%
  summarise(
    Endo_mean = mean(Endothelial_score, na.rm = TRUE),
    Mural_mean = mean(Mural_score, na.rm = TRUE),
    .groups = 'drop'
  )

# 计算总平均（across groups）和差值
hip_score_plot <- hip_scores_summary %>%
  group_by(area_m) %>%
  summarise(
    Endo_overall_mean = mean(Endo_mean),
    Mural_overall_mean = mean(Mural_mean),
    Endo_diff = Endo_mean[group == "AD"] - Endo_mean[group == "Control"],
    Mural_diff = Mural_mean[group == "AD"] - Mural_mean[group == "Control"],
    .groups = 'drop'
  ) %>%
  pivot_longer(
    cols = c(Endo_overall_mean, Mural_overall_mean, Endo_diff, Mural_diff),
    names_to = "variable",
    values_to = "value"
  ) %>%
  mutate(
    cell_type = case_when(
      grepl("Endo", variable) ~ "Endo",
      grepl("Mural", variable) ~ "Mural"
    ),
    metric_type = case_when(
      grepl("overall_mean", variable) ~ "mean",
      grepl("diff", variable) ~ "diff"
    )
  ) %>%
  select(area_m, cell_type, metric_type, value) %>%
  pivot_wider(
    names_from = metric_type,
    values_from = value
  )

# 3. 准备Hip Density数据
hip_density_summary <- hip_density %>%
  group_by(area_m, group, celltype_unit) %>%
  summarise(
    mean_density = mean(density_per_mm2, na.rm = TRUE),
    .groups = 'drop'
  )

hip_density_plot <- hip_density_summary %>%
  group_by(area_m, celltype_unit) %>%
  summarise(
    overall_mean = mean(mean_density),
    diff = mean_density[group == "AD"] - mean_density[group == "Control"],
    .groups = 'drop'
  ) %>%
  rename(cell_type = celltype_unit, mean = overall_mean)

# 4. 准备Cortex Vascular Score数据（使用score_norm的sum）
cortex_scores <- cortex_long %>%
  group_by(sample_id, group, region_merged) %>%
  summarise(
    Endothelial_score = sum(Endothelial_score_norm, na.rm = TRUE),  # 改为sum
    Pericyte_score = sum(Pericyte_score_norm, na.rm = TRUE),        # 改为sum
    SMC_score = sum(SMC_score_norm, na.rm = TRUE),                  # 改为sum
    .groups = 'drop'
  ) %>%
  mutate(Mural_score = Endothelial_score + Pericyte_score + SMC_score)  # 三者之和

cortex_scores_summary <- cortex_scores %>%
  group_by(region_merged, group) %>%
  summarise(
    Endo_mean = mean(Endothelial_score, na.rm = TRUE),
    Mural_mean = mean(Mural_score, na.rm = TRUE),
    .groups = 'drop'
  )

cortex_score_plot <- cortex_scores_summary %>%
  group_by(region_merged) %>%
  summarise(
    Endo_overall_mean = mean(Endo_mean),
    Mural_overall_mean = mean(Mural_mean),
    Endo_diff = Endo_mean[group == "AD"] - Endo_mean[group == "Control"],
    Mural_diff = Mural_mean[group == "AD"] - Mural_mean[group == "Control"],
    .groups = 'drop'
  ) %>%
  pivot_longer(
    cols = c(Endo_overall_mean, Mural_overall_mean, Endo_diff, Mural_diff),
    names_to = "variable",
    values_to = "value"
  ) %>%
  mutate(
    cell_type = case_when(
      grepl("Endo", variable) ~ "Endo",
      grepl("Mural", variable) ~ "Mural"
    ),
    metric_type = case_when(
      grepl("overall_mean", variable) ~ "mean",
      grepl("diff", variable) ~ "diff"
    )
  ) %>%
  select(region_merged, cell_type, metric_type, value) %>%
  pivot_wider(
    names_from = metric_type,
    values_from = value
  )

# 5. 准备Cortex Density数据
cortex_density_summary <- cortex_density %>%
  group_by(region_merged, group, celltype_unit) %>%
  summarise(
    mean_density = mean(density_per_mm2, na.rm = TRUE),
    .groups = 'drop'
  )

cortex_density_plot <- cortex_density_summary %>%
  group_by(region_merged, celltype_unit) %>%
  summarise(
    overall_mean = mean(mean_density),
    diff = mean_density[group == "AD"] - mean_density[group == "Control"],
    .groups = 'drop'
  ) %>%
  rename(cell_type = celltype_unit, mean = overall_mean)

# 6. 主题设置
theme_dotplot <- function(base_size = 12) {
  theme_classic(base_size = base_size) +
    theme(
      plot.title = element_text(face = "bold", size = rel(1.2), hjust = 0.5),
      axis.title = element_text(face = "bold", size = rel(1)),
      axis.text = element_text(size = rel(0.9), color = "black"),
      axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1),
      axis.line = element_line(colour = "black", linewidth = 0.5),
      axis.ticks = element_line(colour = "black", linewidth = 0.5),
      legend.title = element_text(face = "bold", size = rel(0.9)),
      legend.text = element_text(size = rel(0.8)),
      legend.position = "right",
      panel.background = element_blank(),
      panel.grid.major = element_line(color = "grey90", linewidth = 0.2),
      panel.border = element_blank()
    )
}

# 7. 绘制Vascular Score Dotplot
# Hip Vascular Score
hip_areas <- c('CA1', 'CA2', 'CA3', 'CA4', 'DG', 'FAS', 'SLRM')
hip_score_plot$area_m <- factor(hip_score_plot$area_m, levels = hip_areas)
hip_score_plot$cell_type <- factor(hip_score_plot$cell_type, levels = c("Endo", "Mural"))

# 计算差值范围用于颜色
score_diff_max <- max(abs(c(hip_score_plot$diff, cortex_score_plot$diff)), na.rm = TRUE)

p_hip_score <- ggplot(hip_score_plot, aes(x = area_m, y = cell_type)) +
  geom_point(aes(size = mean, color = diff), alpha = 0.8) +
  scale_size_continuous(
    name = "Mean Sum of\nVascular Score",
    range = c(2, 15)
  ) +
  scale_color_gradient2(
    name = "Score Difference\n(AD - Control)",
    low = "#4575B4",
    mid = "white",
    high = "#D73027",
    midpoint = 0,
    limits = c(-score_diff_max, score_diff_max)
  ) +
  labs(
    title = "Hippocampus - Vascular Score\n(Sum of score_norm)",
    x = "Brain Region",
    y = "Cell Type"
  ) +
  theme_dotplot()

# Cortex Vascular Score
cortex_regions <- c('L1', 'L23', 'L456', 'WM')
cortex_score_plot$region_merged <- factor(cortex_score_plot$region_merged, levels = cortex_regions)
cortex_score_plot$cell_type <- factor(cortex_score_plot$cell_type, levels = c("Endo", "Mural"))

p_cortex_score <- ggplot(cortex_score_plot, aes(x = region_merged, y = cell_type)) +
  geom_point(aes(size = mean, color = diff), alpha = 0.8) +
  scale_size_continuous(
    name = "Mean Sum of\nVascular Score",
    range = c(2, 15)
  ) +
  scale_color_gradient2(
    name = "Score Difference\n(AD - Control)",
    low = "#4575B4",
    mid = "white",
    high = "#D73027",
    midpoint = 0,
    limits = c(-score_diff_max, score_diff_max)
  ) +
  labs(
    title = "Cortex - Vascular Score\n(Sum of score_norm)",
    x = "Cortical Layer",
    y = "Cell Type"
  ) +
  theme_dotplot()

# 8. 绘制Density Dotplot
# Hip Density
hip_density_plot$area_m <- factor(hip_density_plot$area_m, levels = hip_areas)
hip_density_plot$cell_type <- factor(hip_density_plot$cell_type, levels = c("Endo", "Mural"))

density_diff_max <- max(abs(c(hip_density_plot$diff, cortex_density_plot$diff)), na.rm = TRUE)

p_hip_density <- ggplot(hip_density_plot, aes(x = area_m, y = cell_type)) +
  geom_point(aes(size = mean, color = diff), alpha = 0.8) +
  scale_size_continuous(
    name = "Mean Density\n(cells/mm²)",
    range = c(2, 15)
  ) +
  scale_color_gradient2(
    name = "Density Difference\n(AD - Control)",
    low = "#4575B4",
    mid = "white",
    high = "#D73027",
    midpoint = 0,
    limits = c(-density_diff_max, density_diff_max)
  ) +
  labs(
    title = "Hippocampus - Cell Density",
    x = "Brain Region",
    y = "Cell Type"
  ) +
  theme_dotplot()

# Cortex Density
cortex_density_plot$region_merged <- factor(cortex_density_plot$region_merged, levels = cortex_regions)
cortex_density_plot$cell_type <- factor(cortex_density_plot$cell_type, levels = c("Endo", "Mural"))

p_cortex_density <- ggplot(cortex_density_plot, aes(x = region_merged, y = cell_type)) +
  geom_point(aes(size = mean, color = diff), alpha = 0.8) +
  scale_size_continuous(
    name = "Mean Density\n(cells/mm²)",
    range = c(2, 15)
  ) +
  scale_color_gradient2(
    name = "Density Difference\n(AD - Control)",
    low = "#4575B4",
    mid = "white",
    high = "#D73027",
    midpoint = 0,
    limits = c(-density_diff_max, density_diff_max)
  ) +
  labs(
    title = "Cortex - Cell Density",
    x = "Cortical Layer",
    y = "Cell Type"
  ) +
  theme_dotplot()

# 9. 保存图片
output_dir <- "../results/Figure1"

# 单独保存
ggsave(file.path(output_dir, "vascular_score_hip_norm_sum.pdf"), 
       p_hip_score, width = 8, height = 3.5, dpi = 300)
ggsave(file.path(output_dir, "vascular_score_cortex_norm_sum.pdf"), 
       p_cortex_score, width = 6, height = 3.5, dpi = 300)
ggsave(file.path(output_dir, "density_hip.pdf"), 
       p_hip_density, width = 8, height = 3, dpi = 300)
ggsave(file.path(output_dir, "density_cortex.pdf"), 
       p_cortex_density, width = 6, height = 3, dpi = 300)

# 组合图 - Vascular Score
p_score_combined <- plot_grid(
  p_hip_score, p_cortex_score,
  labels = c("A", "B"),
  ncol = 2,
  rel_widths = c(1.2, 1)
)

ggsave(file.path(output_dir, "vascular_score_combined_norm_sum.pdf"), 
       p_score_combined, width = 14, height = 3.5, dpi = 300)

# 组合图 - Density
p_density_combined <- plot_grid(
  p_hip_density, p_cortex_density,
  labels = c("A", "B"),
  ncol = 2,
  rel_widths = c(1.2, 1)
)

ggsave(file.path(output_dir, "density_combined.pdf"), 
       p_density_combined, width = 14, height = 3.5, dpi = 300)

# 保存数据
write.csv(hip_score_plot, file.path(output_dir, "hip_score_dotplot_data_norm_sum.csv"), row.names = FALSE)
write.csv(cortex_score_plot, file.path(output_dir, "cortex_score_dotplot_data_norm_sum.csv"), row.names = FALSE)
write.csv(hip_density_plot, file.path(output_dir, "hip_density_dotplot_data.csv"), row.names = FALSE)
write.csv(cortex_density_plot, file.path(output_dir, "cortex_density_dotplot_data.csv"), row.names = FALSE)

print("完成！所有dotplot已保存。")
print(paste0("Vascular Score差值范围: ±", round(score_diff_max, 4)))
print(paste0("Density差值范围: ±", round(density_diff_max, 2)))

In [ ]:
library(tidyverse)
library(ggplot2)
library(cowplot)

# 1. 读取数据
hip_long <- read.csv("../results/01_Ficture/Hip/Hip_all_samples_combined.csv")
cortex_long <- read.csv("../results/01_Ficture/Cortex/Cortex_all_samples_combined.csv")
hip_density <- read.csv('../results/Figure1/Hip_density.csv')
cortex_density <- read.csv('../results/Figure1/Cortex_density.csv')

hip_density <- hip_density[hip_density$area_m %in% c('CA1','CA2','CA3','CA4','FAS','SLRM','DG'),]
hip_long <- hip_long[hip_long$area_m %in% c('CA1','CA2','CA3','CA4','FAS','SLRM','DG'),]
hip_long <- hip_long[!hip_long$sample_id %in% c('D01574C2','D01574C6'),]

# 2. 准备Hip Vascular Score数据（使用score_norm的sum）
hip_scores <- hip_long %>%
  group_by(sample_id, group, area_m) %>%
  summarise(
    Endothelial_score = sum(Endothelial_score_norm, na.rm = TRUE),
    Pericyte_score = sum(Pericyte_score_norm, na.rm = TRUE),
    SMC_score = sum(SMC_score_norm, na.rm = TRUE),
    .groups = 'drop'
  ) %>%
  mutate(Mural_score = Endothelial_score + Pericyte_score)

hip_scores_summary <- hip_scores %>%
  group_by(area_m, group) %>%
  summarise(
    Endo_mean = mean(Endothelial_score, na.rm = TRUE),
    Mural_mean = mean(Mural_score, na.rm = TRUE),
    .groups = 'drop'
  )

hip_score_plot <- hip_scores_summary %>%
  group_by(area_m) %>%
  summarise(
    Endo_overall_mean = mean(Endo_mean),
    Mural_overall_mean = mean(Mural_mean),
    Endo_diff = Endo_mean[group == "AD"] - Endo_mean[group == "Control"],
    Mural_diff = Mural_mean[group == "AD"] - Mural_mean[group == "Control"],
    .groups = 'drop'
  ) %>%
  pivot_longer(
    cols = c(Endo_overall_mean, Mural_overall_mean, Endo_diff, Mural_diff),
    names_to = "variable",
    values_to = "value"
  ) %>%
  mutate(
    cell_type = case_when(
      grepl("Endo", variable) ~ "Endo",
      grepl("Mural", variable) ~ "Mural"
    ),
    metric_type = case_when(
      grepl("overall_mean", variable) ~ "mean",
      grepl("diff", variable) ~ "diff"
    )
  ) %>%
  select(area_m, cell_type, metric_type, value) %>%
  pivot_wider(
    names_from = metric_type,
    values_from = value
  )

# 3. 准备Hip Density数据
hip_density_summary <- hip_density %>%
  group_by(area_m, group, celltype_unit) %>%
  summarise(
    mean_density = mean(density_per_mm2, na.rm = TRUE),
    .groups = 'drop'
  )

hip_density_plot <- hip_density_summary %>%
  group_by(area_m, celltype_unit) %>%
  summarise(
    overall_mean = mean(mean_density),
    diff = mean_density[group == "AD"] - mean_density[group == "Control"],
    .groups = 'drop'
  ) %>%
  rename(cell_type = celltype_unit, mean = overall_mean)

# 4. 准备Cortex Vascular Score数据（使用score_norm的sum）
cortex_scores <- cortex_long %>%
  group_by(sample_id, group, region_merged) %>%
  summarise(
    Endothelial_score = sum(Endothelial_score_norm, na.rm = TRUE),
    Pericyte_score = sum(Pericyte_score_norm, na.rm = TRUE),
    SMC_score = sum(SMC_score_norm, na.rm = TRUE),
    .groups = 'drop'
  ) %>%
  mutate(Mural_score = Endothelial_score + Pericyte_score + SMC_score)

cortex_scores_summary <- cortex_scores %>%
  group_by(region_merged, group) %>%
  summarise(
    Endo_mean = mean(Endothelial_score, na.rm = TRUE),
    Mural_mean = mean(Mural_score, na.rm = TRUE),
    .groups = 'drop'
  )

cortex_score_plot <- cortex_scores_summary %>%
  group_by(region_merged) %>%
  summarise(
    Endo_overall_mean = mean(Endo_mean),
    Mural_overall_mean = mean(Mural_mean),
    Endo_diff = Endo_mean[group == "AD"] - Endo_mean[group == "Control"],
    Mural_diff = Mural_mean[group == "AD"] - Mural_mean[group == "Control"],
    .groups = 'drop'
  ) %>%
  pivot_longer(
    cols = c(Endo_overall_mean, Mural_overall_mean, Endo_diff, Mural_diff),
    names_to = "variable",
    values_to = "value"
  ) %>%
  mutate(
    cell_type = case_when(
      grepl("Endo", variable) ~ "Endo",
      grepl("Mural", variable) ~ "Mural"
    ),
    metric_type = case_when(
      grepl("overall_mean", variable) ~ "mean",
      grepl("diff", variable) ~ "diff"
    )
  ) %>%
  select(region_merged, cell_type, metric_type, value) %>%
  pivot_wider(
    names_from = metric_type,
    values_from = value
  )

# 5. 准备Cortex Density数据
cortex_density_summary <- cortex_density %>%
  group_by(region_merged, group, celltype_unit) %>%
  summarise(
    mean_density = mean(density_per_mm2, na.rm = TRUE),
    .groups = 'drop'
  )

cortex_density_plot <- cortex_density_summary %>%
  group_by(region_merged, celltype_unit) %>%
  summarise(
    overall_mean = mean(mean_density),
    diff = mean_density[group == "AD"] - mean_density[group == "Control"],
    .groups = 'drop'
  ) %>%
  rename(cell_type = celltype_unit, mean = overall_mean)

# 6. 计算统一的legend范围
# 合并Hip和Cortex的density数据
combined_density <- bind_rows(
  hip_density_plot %>% mutate(region = "Hippocampus"),
  cortex_density_plot %>% rename(area_m = region_merged) %>% mutate(region = "Cortex")
)

# 计算统一的范围
density_mean_range <- range(combined_density$mean, na.rm = TRUE)
density_diff_range <- max(abs(combined_density$diff), na.rm = TRUE)

cat("Density mean range:", density_mean_range[1], "to", density_mean_range[2], "\n")
cat("Density diff range: ±", density_diff_range, "\n")

# Vascular Score的差值范围
score_diff_max <- max(abs(c(hip_score_plot$diff, cortex_score_plot$diff)), na.rm = TRUE)

# 7. 主题设置
theme_dotplot <- function(base_size = 12) {
  theme_classic(base_size = base_size) +
    theme(
      plot.title = element_text(face = "bold", size = rel(1.2), hjust = 0.5),
      axis.title = element_text(face = "bold", size = rel(1)),
      axis.text = element_text(size = rel(0.9), color = "black"),
      axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1),
      axis.line = element_line(colour = "black", linewidth = 0.5),
      axis.ticks = element_line(colour = "black", linewidth = 0.5),
      legend.title = element_text(face = "bold", size = rel(0.9)),
      legend.text = element_text(size = rel(0.8)),
      legend.position = "right",
      panel.background = element_blank(),
      panel.grid.major = element_line(color = "grey90", linewidth = 0.2),
      panel.border = element_blank()
    )
}

# 8. 绘制Vascular Score Dotplot
# Hip Vascular Score
hip_areas <- c('CA1', 'CA2', 'CA3', 'CA4', 'DG', 'FAS', 'SLRM')
hip_score_plot$area_m <- factor(hip_score_plot$area_m, levels = hip_areas)
hip_score_plot$cell_type <- factor(hip_score_plot$cell_type, levels = c("Endo", "Mural"))

p_hip_score <- ggplot(hip_score_plot, aes(x = area_m, y = cell_type)) +
  geom_point(aes(size = mean, color = diff), alpha = 0.8) +
  scale_size_continuous(
    name = "Mean Sum of\nVascular Score",
    range = c(2, 15)
  ) +
  scale_color_gradient2(
    name = "Score Difference\n(AD - Control)",
    low = "#4575B4",
    mid = "white",
    high = "#D73027",
    midpoint = 0,
    limits = c(-score_diff_max, score_diff_max)
  ) +
  labs(
    title = "Hippocampus - Vascular Score",
    x = "Brain Region",
    y = "Cell Type"
  ) +
  theme_dotplot()

# Cortex Vascular Score
cortex_regions <- c('L1', 'L23', 'L456', 'WM')
cortex_score_plot$region_merged <- factor(cortex_score_plot$region_merged, levels = cortex_regions)
cortex_score_plot$cell_type <- factor(cortex_score_plot$cell_type, levels = c("Endo", "Mural"))

p_cortex_score <- ggplot(cortex_score_plot, aes(x = region_merged, y = cell_type)) +
  geom_point(aes(size = mean, color = diff), alpha = 0.8) +
  scale_size_continuous(
    name = "Mean Sum of\nVascular Score",
    range = c(2, 15)
  ) +
  scale_color_gradient2(
    name = "Score Difference\n(AD - Control)",
    low = "#4575B4",
    mid = "white",
    high = "#D73027",
    midpoint = 0,
    limits = c(-score_diff_max, score_diff_max)
  ) +
  labs(
    title = "Cortex - Vascular Score",
    x = "Cortical Layer",
    y = "Cell Type"
  ) +
  theme_dotplot()

# 9. 绘制Density Dotplot（统一legend）
# Hip Density
hip_density_plot$area_m <- factor(hip_density_plot$area_m, levels = hip_areas)
hip_density_plot$cell_type <- factor(hip_density_plot$cell_type, levels = c("Endo", "Mural"))

p_hip_density <- ggplot(hip_density_plot, aes(x = area_m, y = cell_type)) +
  geom_point(aes(size = mean, color = diff), alpha = 0.8) +
  scale_size_continuous(
    name = "Mean Density\n(cells/mm²)",
    range = c(2, 15),
    limits = density_mean_range  # 统一范围
  ) +
  scale_color_gradient2(
    name = "Density Difference\n(AD - Control)",
    low = "#4575B4",
    mid = "white",
    high = "#D73027",
    midpoint = 0,
    limits = c(-density_diff_range, density_diff_range)  # 统一范围
  ) +
  labs(
    title = "Hippocampus - Cell Density",
    x = "Brain Region",
    y = "Cell Type"
  ) +
  theme_dotplot()

# Cortex Density
cortex_density_plot$region_merged <- factor(cortex_density_plot$region_merged, levels = cortex_regions)
cortex_density_plot$cell_type <- factor(cortex_density_plot$cell_type, levels = c("Endo", "Mural"))

p_cortex_density <- ggplot(cortex_density_plot, aes(x = region_merged, y = cell_type)) +
  geom_point(aes(size = mean, color = diff), alpha = 0.8) +
  scale_size_continuous(
    name = "Mean Density\n(cells/mm²)",
    range = c(2, 15),
    limits = density_mean_range  # 统一范围
  ) +
  scale_color_gradient2(
    name = "Density Difference\n(AD - Control)",
    low = "#4575B4",
    mid = "white",
    high = "#D73027",
    midpoint = 0,
    limits = c(-density_diff_range, density_diff_range)  # 统一范围
  ) +
  labs(
    title = "Cortex - Cell Density",
    x = "Cortical Layer",
    y = "Cell Type"
  ) +
  theme_dotplot()

# 10. 保存图片
output_dir <- "../results/Figure1"

# 单独保存
ggsave(file.path(output_dir, "vascular_score_hip_norm_sum.pdf"), 
       p_hip_score, width = 8, height = 3.5, dpi = 300)
ggsave(file.path(output_dir, "vascular_score_cortex_norm_sum.pdf"), 
       p_cortex_score, width = 6, height = 3.5, dpi = 300)
ggsave(file.path(output_dir, "density_hip.pdf"), 
       p_hip_density, width = 8, height = 3.5, dpi = 300)
ggsave(file.path(output_dir, "density_cortex.pdf"), 
       p_cortex_density, width = 6, height = 3.5, dpi = 300)

# 组合图 - Vascular Score (左右排布)
p_score_combined <- plot_grid(
  p_hip_score, p_cortex_score,
  labels = c("A", "B"),
  ncol = 2,
  rel_widths = c(1.2, 1)
)

ggsave(file.path(output_dir, "vascular_score_combined_norm_sum.pdf"), 
       p_score_combined, width = 14, height = 4, dpi = 300)

# 组合图 - Density (上下排布，统一legend)
p_density_combined <- plot_grid(
  p_hip_density, p_cortex_density,
  labels = c("A", "B"),
  ncol = 1,  # 改为上下排布
  nrow = 2
)

ggsave(file.path(output_dir, "density_combined_vertical.pdf"), 
       p_density_combined, width = 10, height = 8, dpi = 300)

# 保存数据
write.csv(hip_score_plot, file.path(output_dir, "hip_score_dotplot_data_norm_sum.csv"), row.names = FALSE)
write.csv(cortex_score_plot, file.path(output_dir, "cortex_score_dotplot_data_norm_sum.csv"), row.names = FALSE)
write.csv(hip_density_plot, file.path(output_dir, "hip_density_dotplot_data.csv"), row.names = FALSE)
write.csv(cortex_density_plot, file.path(output_dir, "cortex_density_dotplot_data.csv"), row.names = FALSE)

print("完成！所有dotplot已保存。")
print(paste0("Vascular Score差值范围: ±", round(score_diff_max, 4)))
print(paste0("Density差值范围: ±", round(density_diff_range, 2)))
print(paste0("Density mean范围: ", round(density_mean_range[1], 2), " - ", round(density_mean_range[2], 2)))

## NVU density statistics

In [ ]:
celldata = readRDS('../results/05_nvu_analysis_results/Hip/200/Hip_combined_metadata.rds')
areadata = read.csv('../data/Hip_region_bin50_summary.csv')
celldata = celldata[celldata$area_m%in%c('CA1','CA2','CA3','CA4','FAS','SLRM','DG'),]
areadata$area_m = areadata$area
areadata = areadata[areadata$area_m%in%c('CA1','CA2','CA3','CA4','FAS','SLRM','DG'),]
# 1. 从 areadata 创建 sample 到 chip 的映射
sample_chip_map = unique(areadata[, c('sample', 'chip')])
celldata$chip = sample_chip_map$chip[match(celldata$sample_id, sample_chip_map$sample)]
celldata$group = ifelse(grepl('^AD', celldata$chip), 'AD', 'Control')

In [ ]:
library(ggplot2)
library(dplyr)
library(ggpubr)
library(rstatix)
# 1. 首先给 celldata 添加 chip 信息
sample_chip_map <- unique(areadata[, c('sample', 'chip')])
celldata$chip <- sample_chip_map$chip[match(celldata$sample_id, sample_chip_map$sample)]

# 2. 筛选出 Matched 的数据
matched_data <- celldata[celldata$isTrue == 'Matched', ]

# 3. 统计每个芯片每个脑区的 unit 个数
matched_data <- matched_data[matched_data$unit_id != '', ]

unit_count <- matched_data %>%
  group_by(chip, area_m) %>%
  summarise(n_units = n_distinct(unit_id), .groups = 'drop')

# 4. 从 areadata 获取每个芯片每个脑区的面积
area_size <- areadata %>%
  group_by(chip, area_m) %>%
  summarise(area_mm2 = first(area_mm2), .groups = 'drop')

# 5. 合并数据并计算密度
result <- unit_count %>%
  left_join(area_size, by = c('chip', 'area_m')) %>%
  mutate(density = n_units / area_mm2)

# 6. 添加 group 信息
result$group <- ifelse(grepl('^AD', result$chip), 'AD', 'Control')

# 7. 过滤掉特定的 chip
result <- result[!result$chip %in% c('Con3.1', 'Con3.2','AD1.1', 'AD1.2'), ]
# result[!result$area_m=='SLRM'&result$chip%in%c('Con6.2','Con6.1'),]
# 定义配色
colors <- c("AD" = "#EC8070", "Control" = "#4DBBD5")

plot_data_density <- result %>%
  group_by(group, area_m) %>%
  summarise(
    mean_density = mean(density, na.rm = TRUE),
    sd_density = sd(density, na.rm = TRUE),
    se_density = sd(density, na.rm = TRUE) / sqrt(n()),
    n = n(),
    .groups = 'drop'
  )

plot_data_density$group <- factor(plot_data_density$group, levels = c('Control','AD'))

p1 <- ggplot(plot_data_density, aes(x = area_m, y = mean_density, fill = group)) +
  geom_bar(stat = "identity", position = position_dodge(0.8), 
           width = 0.7, color = "black", linewidth = 0.3) +
  geom_errorbar(aes(ymin = mean_density - se_density, 
                    ymax = mean_density + se_density),
                position = position_dodge(0.8), width = 0.25, 
                linewidth = 0.4) +
  scale_fill_manual(values = colors) +
  labs(x = "Brain Region", 
       y = expression("Unit Density (units/mm"^2*")"),
       fill = "Group") +
  theme_classic(base_size = 12) +
  theme(
    axis.line = element_line(color = "black", linewidth = 0.5),
    axis.ticks = element_line(color = "black", linewidth = 0.5),
    axis.text = element_text(color = "black", size = 11),
    axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1),
    axis.title = element_text(color = "black", size = 13, face = "bold"),
    legend.position = "top",
    legend.title = element_text(size = 12, face = "bold"),
    legend.text = element_text(size = 11),
    legend.key.size = unit(0.5, "cm"),
    panel.grid.major = element_blank(),
    plot.margin = margin(10, 10, 10, 10)
  )

print(p1)
ggsave("unit_density_by_region.pdf", p1, width = 8, height = 6, dpi = 300)

plot_data_count <- result %>%
  group_by(group, area_m) %>%
  summarise(
    mean_count = mean(n_units, na.rm = TRUE),
    sd_count = sd(n_units, na.rm = TRUE),
    se_count = sd(n_units, na.rm = TRUE) / sqrt(n()),
    n = n(),
    .groups = 'drop'
  )

plot_data_count$group <- factor(plot_data_count$group, levels = c('Control','AD'))

p2 <- ggplot(plot_data_count, aes(x = area_m, y = mean_count, fill = group)) +
  geom_bar(stat = "identity", position = position_dodge(0.8), 
           width = 0.7, color = "black", linewidth = 0.3) +
  geom_errorbar(aes(ymin = mean_count - se_count, 
                    ymax = mean_count + se_count),
                position = position_dodge(0.8), width = 0.25, 
                linewidth = 0.4) +
  scale_fill_manual(values = colors) +
  labs(x = "Brain Region", 
       y = "Number of Units",
       fill = "Group") +
  theme_classic(base_size = 12) +
  theme(
    axis.line = element_line(color = "black", linewidth = 0.5),
    axis.ticks = element_line(color = "black", linewidth = 0.5),
    axis.text = element_text(color = "black", size = 11),
    axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1),
    axis.title = element_text(color = "black", size = 13, face = "bold"),
    legend.position = "top",
    legend.title = element_text(size = 12, face = "bold"),
    legend.text = element_text(size = 11),
    legend.key.size = unit(0.5, "cm"),
    panel.grid.major = element_blank(),
    plot.margin = margin(10, 10, 10, 10)
  )

print(p2)
ggsave("unit_count_by_region.pdf", p2, width = 8, height = 6, dpi = 300)

# 计算统计检验
stat_test_density <- result %>%
  group_by(area_m) %>%
  # wilcox_test(density ~ group) %>%
  t_test(density ~ group) %>%
  adjust_pvalue(method = "BH") %>%
  add_significance("p.adj")

# 添加显示位置 - 使用实际的y位置
y_max <- max(plot_data_density$mean_density + plot_data_density$se_density, na.rm = TRUE)
stat_test_density <- stat_test_density %>%
  mutate(
    y.position = y_max * 1.15,  # 在最大值上方15%的位置
    # 添加group1和group2以便正确显示
    group1 = "Control",
    group2 = "AD"
  )

# 方法1: 使用精确p值
p3_pvalue <- ggplot(plot_data_density, aes(x = area_m, y = mean_density, fill = group)) +
  geom_bar(stat = "identity", position = position_dodge(0.8), 
           width = 0.7, color = "black", linewidth = 0.3) +
  geom_errorbar(aes(ymin = mean_density - se_density, 
                    ymax = mean_density + se_density),
                position = position_dodge(0.8), width = 0.25, 
                linewidth = 0.4) +
  geom_text(data = stat_test_density,
            aes(x = area_m, y = y.position, 
                label = sprintf("p = %.3f", p.adj)),
            inherit.aes = FALSE,
            size = 3.5, vjust = -0.5) +
  scale_fill_manual(values = colors) +
  scale_y_continuous(expand = expansion(mult = c(0.05, 0.15))) +
  labs(x = "Brain Region", 
       y = expression("Unit Density (units/mm"^2*")"),
       fill = "Group") +
  theme_classic(base_size = 12) +
  theme(
    axis.line = element_line(color = "black", linewidth = 0.5),
    axis.ticks = element_line(color = "black", linewidth = 0.5),
    axis.text = element_text(color = "black", size = 11),
    axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1),
    axis.title = element_text(color = "black", size = 13, face = "bold"),
    legend.position = "top",
    legend.title = element_text(size = 12, face = "bold"),
    legend.text = element_text(size = 11),
    panel.grid.major = element_blank(),
    plot.margin = margin(20, 10, 10, 10)
  )

print(p3_pvalue)
ggsave("unit_density_with_pvalue.pdf", p3_pvalue, width = 9, height = 7, dpi = 300)


# 计算统计检验
stat_test_count <- result %>%
  group_by(area_m) %>%
  # wilcox_test(n_units ~ group) %>%
 t_test(density ~ group) %>%
  adjust_pvalue(method = "BH") %>%
  add_significance("p.adj")

# 添加显示位置
y_max_count <- max(plot_data_count$mean_count + plot_data_count$se_count, na.rm = TRUE)
stat_test_count <- stat_test_count %>%
  mutate(
    y.position = y_max_count * 1.15,
    group1 = "Control",
    group2 = "AD"
  )

# 方法1: 使用精确p值
p4_pvalue <- ggplot(plot_data_count, aes(x = area_m, y = mean_count, fill = group)) +
  geom_bar(stat = "identity", position = position_dodge(0.8), 
           width = 0.7, color = "black", linewidth = 0.3) +
  geom_errorbar(aes(ymin = mean_count - se_count, 
                    ymax = mean_count + se_count),
                position = position_dodge(0.8), width = 0.25, 
                linewidth = 0.4) +
  geom_text(data = stat_test_count,
            aes(x = area_m, y = y.position, 
                label = sprintf("p = %.3f", p.adj)),
            inherit.aes = FALSE,
            size = 3.5, vjust = -0.5) +
  scale_fill_manual(values = colors) +
  scale_y_continuous(expand = expansion(mult = c(0.05, 0.15))) +
  labs(x = "Brain Region", 
       y = "Number of Units",
       fill = "Group") +
  theme_classic(base_size = 12) +
  theme(
    axis.line = element_line(color = "black", linewidth = 0.5),
    axis.ticks = element_line(color = "black", linewidth = 0.5),
    axis.text = element_text(color = "black", size = 11),
    axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1),
    axis.title = element_text(color = "black", size = 13, face = "bold"),
    legend.position = "top",
    legend.title = element_text(size = 12, face = "bold"),
    legend.text = element_text(size = 11),
    panel.grid.major = element_blank(),
    plot.margin = margin(20, 10, 10, 10)
  )

print(p4_pvalue)
ggsave("unit_count_with_pvalue.pdf", p4_pvalue, width = 9, height = 7, dpi = 300)

cat("\n=== Density Statistics ===\n")
print(stat_test_density %>% select(area_m, p, p.adj, p.adj.signif))

cat("\n=== Count Statistics ===\n")
print(stat_test_count %>% select(area_m, p, p.adj, p.adj.signif))

# 查看汇总统计
cat("\n=== Summary Statistics (Density) ===\n")
print(plot_data_density)

cat("\n=== Summary Statistics (Count) ===\n")
print(plot_data_count)

# 保存统计结果
write.csv(stat_test_density, "density_statistics.csv", row.names = FALSE)
write.csv(stat_test_count, "count_statistics.csv", row.names = FALSE)


In [ ]:
# 汇总所有脑区数据（不区分亚区）
result_all <- result %>%
  group_by(chip, group) %>%
  summarise(density = mean(density, na.rm = TRUE), .groups = 'drop')

# 统计检验
stat_test_all <- result_all %>%
  t_test(density ~ group, ref.group = "Control") %>%
  add_significance("p")

# 汇总均值±SE
plot_data_all <- result_all %>%
  group_by(group) %>%
  summarise(
    mean_density = mean(density, na.rm = TRUE),
    se_density = sd(density, na.rm = TRUE) / sqrt(n()),
    .groups = 'drop'
  ) %>%
  mutate(group = factor(group, levels = c('Control', 'AD')))

y_max_all <- max(plot_data_all$mean_density + plot_data_all$se_density)

stat_test_all <- stat_test_all %>%
  mutate(
    group1 = "Control",
    group2 = "AD",
    y.position = y_max_all * 1.2
  )

# 完全绕开stat_pvalue_manual，改用annotate手动添加显著性标注
p5 <- ggplot(plot_data_all, aes(x = group, y = mean_density, fill = group)) +
  geom_bar(stat = "identity", width = 0.5, color = "black", linewidth = 0.3) +
  geom_errorbar(aes(ymin = mean_density - se_density,
                    ymax = mean_density + se_density),
                width = 0.2, linewidth = 0.4) +
  # 手动添加显著性线和标签
  annotate("segment", x = 1, xend = 2, 
           y = y_max_all * 1.15, yend = y_max_all * 1.15, linewidth = 0.5) +
  annotate("segment", x = 1, xend = 1, 
           y = y_max_all * 1.15, yend = y_max_all * 1.12, linewidth = 0.5) +
  annotate("segment", x = 2, xend = 2, 
           y = y_max_all * 1.15, yend = y_max_all * 1.12, linewidth = 0.5) +
  annotate("text", x = 1.5, y = y_max_all * 1.18, 
           label = "p = 0.026", size = 4) +
  scale_fill_manual(values = colors) +
  scale_y_continuous(expand = expansion(mult = c(0.05, 0.2))) +
  labs(x = "Group",
       y = expression("Unit Density (units/mm"^2*")"),
       fill = "Group") +
  theme_classic(base_size = 12) +
  theme(
    axis.line = element_line(color = "black", linewidth = 0.5),
    axis.ticks = element_line(color = "black", linewidth = 0.5),
    axis.text = element_text(color = "black", size = 11),
    axis.title = element_text(color = "black", size = 13, face = "bold"),
    legend.position = "none",
    plot.margin = margin(20, 10, 10, 10)
  )
p5

print(p5)
ggsave("Hip_unit_density_overall.pdf", p5, width = 4, height = 6, dpi = 300)

cat("\n=== Overall Density Statistics ===\n")
print(stat_test_all %>% select(group1, group2, p, p.signif))

In [ ]:
library(dplyr)
library(ggplot2)
library(ggpubr)
library(rstatix)
library(tibble)

# 读取数据
celldata <- readRDS('../results/05_nvu_analysis_results/Cortex/300/Cortex_combined_metadata.rds')
areadata <- read.csv('../data/Cortex_region_bin50_summary.csv')

# 1. 脑区合并映射
region_merge_map <- tibble(
  region = c("L1", "L2", "L23", "L3", "L4", "L45", "L456", "L5", "L6", "WM"),
  region_merged = c(
    "L1", "L23", "L23", "L23", "L456", "L456", "L456", "L456", "L456", "WM"
  )
)

# 2. 对 celldata 应用脑区合并
celldata <- celldata %>%
  left_join(region_merge_map, by = c("area" = "region")) %>%
  mutate(area_m = ifelse(is.na(region_merged), area, region_merged))

# 3. 对 areadata 应用脑区合并并合并面积
areadata <- areadata %>%
  left_join(region_merge_map, by = c("region" = "region")) %>%
  mutate(area_m = ifelse(is.na(region_merged), region, region_merged)) %>%
  group_by(group, sample, area_m) %>%
  summarise(
    bin50_count = sum(bin50_count),
    area_mm2 = sum(area_mm2),
    .groups = 'drop'
  )

# 4. 筛选出 Matched 的数据并移除空的 unit_id
matched_data <- celldata %>%
  filter(isTrue == 'Matched', unit_id != '')

# 5. 统计每个样本每个脑区的 unit 个数
unit_count <- matched_data %>%
  group_by(sample_id, area_m) %>%
  summarise(n_units = n_distinct(unit_id), .groups = 'drop')

# 6. 从 areadata 获取每个样本每个脑区的面积和组信息
area_size <- areadata %>%
  select(sample, area_m, area_mm2, group) %>%
  distinct()

# 7. 合并数据并计算密度
result <- unit_count %>%
  left_join(area_size, by = c("sample_id" = "sample", "area_m" = "area_m")) %>%
  mutate(density = n_units / area_mm2) %>%
  filter(!is.na(group))

# 过滤特定样本（如果需要）
result <- result %>% filter(!sample_id %in% c('GSM8330064_B02008C6','GSM8330063_A02092E1'))

# 检查样本匹配情况
cat("\n=== Sample matching check ===\n")
cat("Total observations:", nrow(result), "\n")
cat("Unique samples in celldata:", length(unique(celldata$sample_id)), "\n")
cat("Unique samples in areadata:", length(unique(areadata$sample)), "\n")
cat("Matched samples in result:", length(unique(result$sample_id)), "\n")

summary_stats <- result %>%
  group_by(group, area_m) %>%
  summarise(
    mean_density = mean(density, na.rm = TRUE),
    sd_density = sd(density, na.rm = TRUE),
    se_density = sd(density, na.rm = TRUE) / sqrt(n()),
    mean_count = mean(n_units, na.rm = TRUE),
    sd_count = sd(n_units, na.rm = TRUE),
    se_count = sd(n_units, na.rm = TRUE) / sqrt(n()),
    n = n(),
    .groups = 'drop'
  )

cat("\n=== Summary Statistics ===\n")
print(summary_stats)

plot_data_density <- result %>%
  group_by(group, area_m) %>%
  summarise(
    mean_density = mean(density, na.rm = TRUE),
    se_density = sd(density, na.rm = TRUE) / sqrt(n()),
    n = n(),
    .groups = 'drop'
  ) %>%
  mutate(group = factor(group, levels = c('Control', 'AD')))

# 统计检验
stat_test_density <- result %>%
  group_by(area_m) %>%
  wilcox_test(density ~ group) %>%
  adjust_pvalue(method = "BH") %>%
  add_significance("p.adj")

# 添加显示位置
y_max_density <- max(plot_data_density$mean_density + plot_data_density$se_density, na.rm = TRUE)
stat_test_density <- stat_test_density %>%
  mutate(
    y.position = y_max_density * 1.15,
    group1 = "Control",
    group2 = "AD"
  )

# 配色
colors <- c("AD" = "#E64B35", "Control" = "#4DBBD5")

# 密度图 - 精确p值版本
p1 <- ggplot(plot_data_density, aes(x = area_m, y = mean_density, fill = group)) +
  geom_bar(stat = "identity", position = position_dodge(0.8), 
           width = 0.7, color = "black", linewidth = 0.3) +
  geom_errorbar(aes(ymin = mean_density - se_density, 
                    ymax = mean_density + se_density),
                position = position_dodge(0.8), width = 0.25, 
                linewidth = 0.4) +
  geom_text(data = stat_test_density,
            aes(x = area_m, y = y.position, 
                label = sprintf("p = %.3f", p)),
            inherit.aes = FALSE,
            size = 3.5, vjust = -0.5) +
  scale_fill_manual(values = colors) +
  scale_y_continuous(expand = expansion(mult = c(0.05, 0.15))) +
  labs(x = "Brain Region", 
       y = expression("Unit Density (units/mm"^2*")"),
       fill = "Group",
       title = "Cortex Unit Density by Region") +
  theme_classic(base_size = 12) +
  theme(
    axis.line = element_line(color = "black", linewidth = 0.5),
    axis.ticks = element_line(color = "black", linewidth = 0.5),
    axis.text = element_text(color = "black", size = 11),
    axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1, face = "bold"),
    axis.title = element_text(color = "black", size = 13, face = "bold"),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    legend.position = "top",
    legend.title = element_text(size = 12, face = "bold"),
    legend.text = element_text(size = 11),
    panel.grid.major = element_blank(),
    plot.margin = margin(20, 10, 10, 10)
  )

print(p1)
ggsave("cortex_unit_density_with_pvalue.pdf", p1, width = 8, height = 7, dpi = 300)
ggsave("cortex_unit_density_with_pvalue.png", p1, width = 8, height = 7, dpi = 300)

# 密度图 - 显著性符号版本（可选）
p1_signif <- ggplot(plot_data_density, aes(x = area_m, y = mean_density, fill = group)) +
  geom_bar(stat = "identity", position = position_dodge(0.8), 
           width = 0.7, color = "black", linewidth = 0.3) +
  geom_errorbar(aes(ymin = mean_density - se_density, 
                    ymax = mean_density + se_density),
                position = position_dodge(0.8), width = 0.25, 
                linewidth = 0.4) +
  geom_text(data = stat_test_density,
            aes(x = area_m, y = y.position, label = p.adj.signif),
            inherit.aes = FALSE,
            size = 5, vjust = -0.5) +
  scale_fill_manual(values = colors) +
  scale_y_continuous(expand = expansion(mult = c(0.05, 0.15))) +
  labs(x = "Brain Region", 
       y = expression("Unit Density (units/mm"^2*")"),
       fill = "Group",
       title = "Cortex Unit Density by Region") +
  theme_classic(base_size = 12) +
  theme(
    axis.line = element_line(color = "black", linewidth = 0.5),
    axis.ticks = element_line(color = "black", linewidth = 0.5),
    axis.text = element_text(color = "black", size = 11),
    axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1, face = "bold"),
    axis.title = element_text(color = "black", size = 13, face = "bold"),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    legend.position = "top",
    legend.title = element_text(size = 12, face = "bold"),
    legend.text = element_text(size = 11),
    panel.grid.major = element_blank(),
    plot.margin = margin(20, 10, 10, 10)
  )

ggsave("cortex_unit_density_with_signif.pdf", p1_signif, width = 8, height = 7, dpi = 300)

plot_data_count <- result %>%
  group_by(group, area_m) %>%
  summarise(
    mean_count = mean(n_units, na.rm = TRUE),
    se_count = sd(n_units, na.rm = TRUE) / sqrt(n()),
    n = n(),
    .groups = 'drop'
  ) %>%
  mutate(group = factor(group, levels = c('Control', 'AD')))

# 统计检验
stat_test_count <- result %>%
  group_by(area_m) %>%
  wilcox_test(n_units ~ group) %>%
  adjust_pvalue(method = "BH") %>%
  add_significance("p.adj")

# 添加显示位置
y_max_count <- max(plot_data_count$mean_count + plot_data_count$se_count, na.rm = TRUE)
stat_test_count <- stat_test_count %>%
  mutate(
    y.position = y_max_count * 1.15,
    group1 = "Control",
    group2 = "AD"
  )

# 数量图 - 精确p值版本
p2 <- ggplot(plot_data_count, aes(x = area_m, y = mean_count, fill = group)) +
  geom_bar(stat = "identity", position = position_dodge(0.8), 
           width = 0.7, color = "black", linewidth = 0.3) +
  geom_errorbar(aes(ymin = mean_count - se_count, 
                    ymax = mean_count + se_count),
                position = position_dodge(0.8), width = 0.25, 
                linewidth = 0.4) +
  geom_text(data = stat_test_count,
            aes(x = area_m, y = y.position, 
                label = sprintf("p = %.3f", p.adj)),
            inherit.aes = FALSE,
            size = 3.5, vjust = -0.5) +
  scale_fill_manual(values = colors) +
  scale_y_continuous(expand = expansion(mult = c(0.05, 0.15))) +
  labs(x = "Brain Region", 
       y = "Number of Units",
       fill = "Group",
       title = "Cortex Unit Count by Region") +
  theme_classic(base_size = 12) +
  theme(
    axis.line = element_line(color = "black", linewidth = 0.5),
    axis.ticks = element_line(color = "black", linewidth = 0.5),
    axis.text = element_text(color = "black", size = 11),
    axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1, face = "bold"),
    axis.title = element_text(color = "black", size = 13, face = "bold"),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    legend.position = "top",
    legend.title = element_text(size = 12, face = "bold"),
    legend.text = element_text(size = 11),
    panel.grid.major = element_blank(),
    plot.margin = margin(20, 10, 10, 10)
  )

print(p2)
ggsave("cortex_unit_count_with_pvalue.pdf", p2, width = 8, height = 7, dpi = 300)
ggsave("cortex_unit_count_with_pvalue.png", p2, width = 8, height = 7, dpi = 300)

# 数量图 - 显著性符号版本（可选）
p2_signif <- ggplot(plot_data_count, aes(x = area_m, y = mean_count, fill = group)) +
  geom_bar(stat = "identity", position = position_dodge(0.8), 
           width = 0.7, color = "black", linewidth = 0.3) +
  geom_errorbar(aes(ymin = mean_count - se_count, 
                    ymax = mean_count + se_count),
                position = position_dodge(0.8), width = 0.25, 
                linewidth = 0.4) +
  geom_text(data = stat_test_count,
            aes(x = area_m, y = y.position, label = p.adj.signif),
            inherit.aes = FALSE,
            size = 5, vjust = -0.5) +
  scale_fill_manual(values = colors) +
  scale_y_continuous(expand = expansion(mult = c(0.05, 0.15))) +
  labs(x = "Brain Region", 
       y = "Number of Units",
       fill = "Group",
       title = "Cortex Unit Count by Region") +
  theme_classic(base_size = 12) +
  theme(
    axis.line = element_line(color = "black", linewidth = 0.5),
    axis.ticks = element_line(color = "black", linewidth = 0.5),
    axis.text = element_text(color = "black", size = 11),
    axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1, face = "bold"),
    axis.title = element_text(color = "black", size = 13, face = "bold"),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    legend.position = "top",
    legend.title = element_text(size = 12, face = "bold"),
    legend.text = element_text(size = 11),
    panel.grid.major = element_blank(),
    plot.margin = margin(20, 10, 10, 10)
  )

ggsave("cortex_unit_count_with_signif.pdf", p2_signif, width = 8, height = 7, dpi = 300)

cat("\n=== Density Statistics ===\n")
print(stat_test_density %>% select(area_m, p, p.adj, p.adj.signif))

cat("\n=== Count Statistics ===\n")
print(stat_test_count %>% select(area_m, p, p.adj, p.adj.signif))

# 查看每组的样本数
cat("\n=== Sample sizes per group ===\n")
result %>%
  group_by(area_m, group) %>%
  summarise(n = n(), .groups = 'drop') %>%
  print()

# 保存统计结果
write.csv(stat_test_density, "cortex_density_statistics.csv", row.names = FALSE)
write.csv(stat_test_count, "cortex_count_statistics.csv", row.names = FALSE)
write.csv(summary_stats, "cortex_summary_statistics.csv", row.names = FALSE)
write.csv(result, "cortex_raw_data.csv", row.names = FALSE)

cat("- cortex_unit_density_with_pvalue.pdf/png (精确p值)\n")
cat("- cortex_unit_density_with_signif.pdf (显著性符号)\n")
cat("- cortex_unit_count_with_pvalue.pdf/png (精确p值)\n")
cat("- cortex_unit_count_with_signif.pdf (显著性符号)\n")
cat("- cortex_density_statistics.csv\n")
cat("- cortex_count_statistics.csv\n")
cat("- cortex_summary_statistics.csv\n")
cat("- cortex_raw_data.csv\n")

In [ ]:
# 汇总所有脑区数据（不区分亚区），按sample_id取均值
result_all <- result %>%
  group_by(sample_id, group) %>%
  summarise(density = mean(density, na.rm = TRUE), .groups = 'drop')

# 统计检验
stat_test_all <- result_all %>%
  t_test(density ~ group, ref.group = "Control") %>%
  add_significance("p")

# 汇总均值±SE
plot_data_all <- result_all %>%
  group_by(group) %>%
  summarise(
    mean_density = mean(density, na.rm = TRUE),
    se_density = sd(density, na.rm = TRUE) / sqrt(n()),
    .groups = 'drop'
  ) %>%
  mutate(group = factor(group, levels = c('Control', 'AD')))

y_max_all <- max(plot_data_all$mean_density + plot_data_all$se_density)
p_label <- paste0("p = ", stat_test_all$p[1])

p5 <- ggplot(plot_data_all, aes(x = group, y = mean_density, fill = group)) +
  geom_bar(stat = "identity", width = 0.5, color = "black", linewidth = 0.3) +
  geom_errorbar(aes(ymin = mean_density - se_density,
                    ymax = mean_density + se_density),
                width = 0.2, linewidth = 0.4) +
  # geom_jitter(data = result_all, aes(x = group, y = density),
  #             inherit.aes = FALSE,
  #             width = 0.08, size = 1.5, alpha = 0.6, color = "black") +
  # 手动添加显著性标注
  annotate("segment", x = 1, xend = 2,
           y = y_max_all * 1.15, yend = y_max_all * 1.15, linewidth = 0.5) +
  annotate("segment", x = 1, xend = 1,
           y = y_max_all * 1.15, yend = y_max_all * 1.12, linewidth = 0.5) +
  annotate("segment", x = 2, xend = 2,
           y = y_max_all * 1.15, yend = y_max_all * 1.12, linewidth = 0.5) +
  annotate("text", x = 1.5, y = y_max_all * 1.19,
           label = p_label, size = 4) +
  scale_fill_manual(values = colors) +
  scale_y_continuous(expand = expansion(mult = c(0.05, 0.25))) +
  labs(x = "Group",
       y = expression("Unit Density (units/mm"^2*")"),
       fill = "Group",
       title = "Cortex Overall Unit Density") +
  theme_classic(base_size = 12) +
  theme(
    axis.line = element_line(color = "black", linewidth = 0.5),
    axis.ticks = element_line(color = "black", linewidth = 0.5),
    axis.text = element_text(color = "black", size = 11),
    axis.title = element_text(color = "black", size = 13, face = "bold"),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    legend.position = "none",
    plot.margin = margin(20, 10, 10, 10)
  )

print(p5)
ggsave("Cortex_unit_density_overall.pdf", p5, width = 4, height = 6, dpi = 300)